# HEART H200 — One-Notebook evaluation and report figures

**One `.ipynb` file only**. It reads the existing original ligand LMDB and pretrained H200 `checkpoint_best.pt`, and reproduces the 192-sample selection before expanding to 10,000 **validation** molecules. All Python implementation is embedded below; no helper `.py` and no bundled `.csv` needed. **No training, no Slurm, no writes to the existing model/checkpoint/LMDB.**

**Workstation path policy**: notebook/code on `/home/ypan/...`; generated audits/arrays/figures/temp on `/project/yli4/ypan/heart_evaluation/general_audit_10k_v2/`. Historical training/validation curves are read from the original TensorBoard logs or H200 log files if found, with embedded source-derived 510k–1M historical records as a self-contained fallback. They are *not* the same as 10k inference losses; avoid mixing them on one y-axis.

**Scientific caution**: falling validation loss with no upward divergence is consistent with *no obvious overfitting in the logged phase*, not proof of no overfitting. Check duplicated train/valid SMILES and compare the losses from the original train/valid evaluation settings. The 10k diagnostic uses 8-molecule contrastive batches and cannot be directly compared to historic batch-448 criterion losses.

## 0. Paths and one-time checks

Use the `(heart_gpu)` kernel. Run this cell first. Paths come from the HPC terminal screenshots; checkpoint prefers the original project run location.

In [ ]:
from pathlib import Path
import os, sys, json, warnings, subprocess, re, io, zlib, base64, pickle
from argparse import Namespace
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

HOME_BASE=Path('/home/ypan/heart_liver/model_train')
PROJECT_BASE=Path('/project/yli4/ypan')
SCRATCH_BASE=Path('/scratch/ypan/heart_liver/model_train')
RESULTS=PROJECT_BASE/'heart_evaluation'/'general_audit_10k_v2'
AUDIT_DIR=RESULTS/'full_ligand_audit'
MANIFEST_DIR=RESULTS/'validation_10k_seed17'
RUN_ROOT=RESULTS/'checkpoint_evaluation'
DATA_DIR=PROJECT_BASE/'UniMolReference'/'ligand_data'/'ligands'
REPO_CANDIDATES=[HOME_BASE/'heart-molecular-pretraining-v2',SCRATCH_BASE/'heart-molecular-pretraining-v2']
REPO_V2=next((x for x in REPO_CANDIDATES if (x/'model/unimol/tasks/unimol_contrastive.py').is_file()),REPO_CANDIDATES[0])
UNICORE_CANDIDATES=[HOME_BASE/'third_party/Uni-Core',PROJECT_BASE/'third_party/Uni-Core',SCRATCH_BASE/'third_party/Uni-Core']
UNICORE_ROOT=next((x for x in UNICORE_CANDIDATES if x.is_dir()),UNICORE_CANDIDATES[0])
RUN_REL=Path('runs/h200/full/heart-h200-b448-v1')
CHECKPOINT_OVERRIDE=None
CHECKPOINT_CANDIDATES=[PROJECT_BASE/RUN_REL/'checkpoint_best.pt',HOME_BASE/'heart-molecular-pretraining-v2'/RUN_REL/'checkpoint_best.pt',SCRATCH_BASE/'heart-molecular-pretraining-v2'/RUN_REL/'checkpoint_best.pt',HOME_BASE/'heart-molecular-pretraining'/RUN_REL/'checkpoint_best.pt']
CHECKPOINT=Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else next((p for p in CHECKPOINT_CANDIDATES if p.is_file()),None)
for label,path in [('HEART v2',REPO_V2),('Original 192 reference',REPO_V2/'model/HEART_H200_checkpoint_best_embedding_evaluation.ipynb'),('Ligand LMDBs',DATA_DIR),('Uni-Core',UNICORE_ROOT),('Project results',RESULTS)]: print(f'{label:24} {path} | exists={path.exists()}')
print('Checkpoint candidates:')
for p in CHECKPOINT_CANDIDATES:print(' FOUND' if p.is_file() else ' -    ',p)
print('Selected checkpoint:',CHECKPOINT,'\nKernel:',sys.executable)
assert (REPO_V2/'model/unimol/tasks/unimol_contrastive.py').is_file(),'HEART v2 model not found.'
assert all((DATA_DIR/x).is_file() for x in ['train.lmdb','valid.lmdb','dict.txt']),'Ligand LMDB not found in /project/yli4/ypan/UniMolReference/ligand_data/ligands'
assert CHECKPOINT is not None and CHECKPOINT.is_file(),'H200 checkpoint missing. See printed candidates; set CHECKPOINT_OVERRIDE if needed.'
assert RESULTS.is_relative_to(PROJECT_BASE),'Results must be under /project/yli4/ypan.'
RESULTS.mkdir(parents=True,exist_ok=True)

## 1. Historical **training and validation** loss, model selection, and overfitting check

The preferred source is **original TensorBoard event files** (`runs/.../tsb/{train,valid}`); second choice is original log(s); if those files were moved, notebook carries the **510,100–1,000,000-update** historical records from the supplied H200 `train-1013205.log` (49 validation points). All plots and CSV exports are generated by this single notebook, not shipped as separate files. The available training log uses `train_inner` (logged minibatch averages) and full validation snapshots, so direct train/valid gaps can be influenced by aggregation and augmentation differences; interpret cautiously.

In [ ]:
# Immutable logged data extracted from the pre-existing H200 run; fallback only.
_HIST_TRAIN_B85 = 'c-n=%-L57(j2(6#Aa7y>694%$gTdGrL1Itf8Rz+}Q<4h3dl$KG8V6?+Dv6@VwMa_e|M*}3{a^mq|My@2^*{gHfBcvK_uv2HKmN;q|JVQcfBxHl{g;3Jw|~3;`EURJ@Bj0^{>$^nfBX0U@t^<mU;pDj|L_0*zdirs|NK96jycAE8UL5d`X}Ajzhm-0|7uLOM}m$7{T30Kj|guOj$a}kNsoxsA{qRM=$})GKfl8>9|=2y_{Si$zvE{R{ykgPM<VL=%=AYj&v){-$na;BrM6mR$|KUK)}HT7`mvt)5~)@DPDRR5uJ)%kqnz2=50THgT;0ynSY}|6qg=7yBEj*G2#tD_A<uMaLcI{xCg;zBh@07NQ=ranx5<&#rf}9w)-34E_9&P2_lK1BDA9cAOgG_*cvNP24<b*viA?2G-ch(-(6h@d=zQ?i?C-<8%;!-X8(hs*?1zZW{hg@&%ARUA0`~Yo8I4A^p-7|G<Kcv$o0O|p`!MUVN0G*s;#smNo{QBVI(|jG+`>CZp!gz<$;y0mtD|<=+o>xyStWRSl#xa1)$IK3YlR=rq*t^0u^xZardIFI_Dt$ckUHy$I?|oEthH*dHQ9icZ4(AeYs~hJ!Z2W(K%Z_3&xct(%2Ay;jPKTHm*O#P%Vrn%LG%Ij;x$S*YH>nG$Qf$Iv2I-<iu3DjKjvbwM`=L9QJh1c;Mq*&AAuN+6MRfE8mlGRgFwYb6T>8*^@cQta=xPoh=f6dj3$g7pB~YK%QM?-^v2Kb<fyf>PkYWEKEpl?CF#d^;=`<yYaD2_MFd)8Dg-=fXq&U~wmSBApv`8Z+700TfwjNCBY)yr8WZ778W4NYOUU?lnA+P+4Iw)#qS0XleI$O`?FVU~@e|j^-$$7`KDOf}T|QP*P?~idyEB=PGSH+(WN*+*$UYFjhLbcK_;p1TfjD{z8JG5S30b$IXi&Iq)g>Z_bO~9xi8KKaBqD?+iMamCF>JzC`_eHlRZG@*r>!^YScB{}2Ra_qxP)g29u0)p=kz6GAE6<)1VNp2T<qwCr9ME%^ciLmhz-PLdg)jhr06d?_O(5n@QBHPo>XSD=KUcd=$&3dR%`AhWF`2eOUO!2KmxO0)bO#po~Z5mSezvMj*dE6be8-VBa7cSkd*K!4BeMX!(g^!(Z<`}Q4E@;dY{lB4Fn_TGO{oApP6)w?5O`TvYLg`%gCz98UsK>CU_!Ef@XtKGK1UG(6RSJjoF@H*wL}`As;XXT{`x)qT?xFJ~k`FPu{tNect~fWM9zfC1hVvW5*yznFMl&C&&X1l8r~+>**NTGu4<VnMcuJgkuOVBcsysUq<%zp(~<@o=_O{>Mtdu(r3Jstc-_|C_PD8tM*RirDV-01Y=ObJOaIxjLH*#DH%%v7SkY(NQY4HGP1@J5=Jt<l&p5h{8BQjMF{yG{)>`*OqV-Kw)dGw6Unw-cr=mB_jej>yL}vw27+xL#{+;{eks|kd^W$7j6QhkGuM}q{T5(<);l4#<td0t@zi><Ec-*&O!j=pQS{&$C^Ya)l&NMK&{U{#3CC3EhjM?ec|KN4jdITkqRu01<&xIvh{sf3LRMx$xvP&I)l!rH+VPSvBcrU46LaSg9)XEa1b38dgJ0%`?A}qNiMo4}SrMFPw&p|IeRZ7d_$H3V_|J#3hny^(X7`e1LU()_+2<BMID%hBR&C+S$D~}1D9X3;OUQl?awQz$W0ruS;}SCZjaGcvvmhPtoa(GS+&LXeC>_DdjZ@z%Qh}DOHU4_1TPt^CD^{<)%3V;PWF5bZtny9Pgt6mz410u^kIf1zvM~%E*4XX+km3(nJ<8GkrDR`+Y*G711ZEQ0vH`A7K7t+NFQ4EvUqaT6!4NmO)E@5GN=L}flHHoj``$!vXu}3Q9%#i0bsqzfNB)bEeSmC!8CeM?J3@BCe+Z(W{1UPcS9%7q#_eyrzI=@011xSl^H+)N191LTA^S)^^2^7lRPE(s^x*@ad^ji7`*R#=GD62G5Rzt-S3H}XP4)4B{g;nf2}LqWxBrS{Bl5sJ>G;@Qk-U7Y$|J~VaP31Hq+?@`YGnv2p~*|ezW6`y<ybw}BjIo+0V1KiRIEBZ(s8lv&t*;&6=V_$KQb9%aG8Q4lyq$DuBh<``~1jk0K)b=GP@wQ2gx5!<x6waDEHVK%_b8OTZGUyUo!UT-_CdBhgQA}judX<OUGtC7e@hAsG$sxAyA@plkh-E9^nZ-!;LqP{S~gu*iR)-7<+?D#XfiEd<S}i%f#qobH>=~4V0-6?r_)kvo9Abaf%2e4AOD2vpvEnSc(V&z9JBLi5S9fBHqxxEz#KQae|U)6JIj+;W3^eE*1OoDN~=-Xy<CT@48g%>#QFqsZDO?s7)2o8&Sh^@p>+2%}2B3r61Lf=WgS-fzI;fVs(Z`E*B%OaH3o;_BGoFzlk-fJ$7w4iWy?L@)pNRbF(Jxb!_$}VkHk2Z}`HQtj~tEZ-4z#F)D+LGxU=#P;r-vHMVl8*w-mLK=D$sFPJ{^*+dWMzr2A6+gL6c`<m^sxJ$<9BcSdik6bd=EGQWIudY0Hw&=34<_txn+_Q@>9rF^dL^L926sX)Bh!5mMTv(D|$jk=X?lA<eCL7J}?YU&^Ymg)1kVR#{!<<>$+xEwLx!9Le-|NvaYs21=Oh@MuY<~#41byjP<q>o!LdQA``4#PIWBYXIPd@hHk@e+cW!TjT-L1}vzB38Ab)8Khd(26vfC`&VtxaWnFP#lz*eF70lbzY;R4*H=Q~Nrb0?wH(4Upqesx^DmM$}ApE1#N1(DNCAmQS$7L&-oENXhFx9=t`ML(r}?wh6UfHumM4&i?R35mhfA``|bhu}9(B;vT23-1yq#zE}|#R&J6ANB!JDgyZ2j5IIq<Gc-p+1~e9F!q_v{zv$S9+JsSxR7FtOR)NKGg|#siuc7a@B?)v)yUN&^>rXQFHIa^uowK2xJ9u!qzJ;;!fT;kiuL@HLss@OOc5RB=Unoj~t7bzAx4qgxtFc*1ob9v5Uhg0<ewiS!s@C-%?4aHdv9o<v+v{`qETBiSL~Rc0&|G~Y!mdtm5JvvU1>jJ9DVUaruc?NeVv2Uvu;cf<PJnN7eF<3Yqhr)|bgU#M%1dhy#4y3APNmT$z|SbPZY|}p$G;fZm$SeOOe0XLS6>EJ(i+W%-yYk~2@zGo&JInk=W@<z3Nr1)ZuVti&4L^qkjugva+Hm3w^r<BVO*3F?ImGbB0nSuAru>oQn@x7m#}Tc2DDonhCrmfBy1KT%m!@iu%;+>BrXU0%waso!!8G#zqMk^Wj!J_oIR#uvq4?kYHbQ2do~+5yZydRL(Mrh9q?$IHR{^BZz<Swe2hk4ld;Rd>TIyR6zumzt|<{Chhc9)tPBlC3Sg#D;|=bLuQ#BI)Y#=<WhTogkz+C*a4I}eveB3tpR%>W4BVS>1H6twz7$L<HfS#et2c-nPkYRo1Nu=;<kIbOurFqs^Vvk1aJo|08#kx4mxW0^h#&-R((&7E{g;J(?ek75n>d24CIJmf5^-(Q27#TqyiBatOUJ~H{VT&2T`Jans6n@P6qKs9w=7m`?yZZ>GGc0?T)Wvo=*q0I+&$a2F1F+0OUJbGVJ7f1ib%U%HumXFX)nj|im}~VLNMY325*O|y^O4xN<+yuN41xdarqfoxUsh?R{OZOD#oSPzLe}!<ViW2V}IM0#kM89WwDRr;kiZXV0Kp&joEI^g&=-tp?E=tY7iEVXB#LXWi2P$_LKu|QBG9gv4o7*UQSkniMKLFA6RZL`Xyy`1W$uVI~#|4M+rO%bh!RkZR~?y_&`0NJ_ya$M*|i1F8WK$$}RGjmQ^<A9V^=g<-<`#^brW}`ACEs{pDpPmiF1Wg|niUmr<#?$@Pkk{-@M)xgSylvixmUtelw7#&EZ>yv4D~hxtp)YVA{_+Ko1xCfsHs6rP=+Q*dfP*g0i{+QOX-&$j8Q+p48#{GjidbkXe5KEq(<JQ0-emzim)6vkHjQnM0N`b*7bdB_+h!JHj#b4ShYq8opERwM(u%9one2VStleYsiHSL7nl<X&o4A2Vs7!X~vYkUhB6?BShm*2w1A<x9=9{E#-Ld*YqKq=G9$SE&tDIKBm!$y^P8$Px5+4V3Wou4nOVqYYFi@xQp4e|T<d!rAs>mz&MsQ?aLz;wFu?>=R;&N^)*Z3Gyh;9J(ljFw6}m$K3}OnSJ%h9k$iEH!k=1Fc-Mwu4nO_AvYLAI(Bxf-vc^shw<zsXR|W6n9WSK+lUT?ZU1z+*;k7F9QeTDEVME0ec&%S>ps-z_9!j_kZ^ywS?!Z^zy)QK3p>h9j1>|_vy^6sw&GIjLAKYx%*-z{`_lda(j7B90VJVr<7H+aSljrKn7<|HPiGsKn9V3$X^bC=&EFdsqf_kpOp_kp!6OC?WbyK{%B*8RUmKT}{a7*mxjiuliXJg1@G3Do1eW_d49HSkQa1f#d!F|oV=A19fx2o;hdVd}(es#zY>B!|%{FS9=#&N_fDkVyqe>^2l6}qf93E0=OJpc6#N}k4O4&O|0nP0tWu?0g9#C;zy==$-cv)HHyMh_Hw_rh}AUaZZMGU3&f+24Rj<UFVjD-`S`H)7p=R1P!RtTQ5wqCZUf(=(t6bmsFPeVFjyKSYRWQSPCrDQ++7ya2)5IRtTD|9voWGhY~{0c>N-sp;DXPZ^4i26>pLkN3}1|S^2-`2{`cfJ4u{V?0h$3Ai3-XJRx!VRfLh!KP_^puy7eNE-KK~ognUp`i2qL+`=ihKE3J;;|wpg$Q-Tsrm{nEhFgPw2Cq&`EVhvHPWC9~f^h#}#FE!>w+tx^s0D-7gvYl2F-`O_aHc%f>z(+&&%A_}b&X0&4b?jnNN>9fD#<z>Ii=p-fzKWbBR-s5%%g8~eS5XURDb-n?9_QGOB}gdv`3x!AVaj*A@$L;NSb#`?zrW@EG`LS+KSl(2I#$3!`}9E7R_IT~EMeSTsxmK_<}=9c5DErEW1$dL!k0duxF>L_-0C}-WPbcj^WiFid$0&X%VBfObYN}uu0_9%C1<pgv7OwW8!>TC$vD-tN#ntu^8|CkH?jAK7U*q#s3L?|MOko{jq_N5li{T+d7xs<FXQ%QzEX}*-KnM>3e=Bt)$HhM3bM-3MDf_TCM%>?{lGD;&POYH}@t4vntH!dOjp%nUY;@JT;&zFzY5)EfKRx1s#J?puA?8`lEzn+(m9g+NnX!6prDWw^RKu>x3*r(q1cMSScfq8@!NE-6xV^$i14n&7jVf`^*J~k_Km6wmz+%OgU+e^rPY|#(ps?}%t*pqr~30Eyq6QwhgM90uZz2Y;7O_aLNj*uP4-x0Da(h#!aoJ^7Xx`gcb!2H-ka~M4?A^YTp+xjh`SN8!jSLrZClb4YFaOyvOk2ADM7wExU5i(8x0dM_7EAG%(UQSkf7XRQ4^hdrTV}NXG#zYAD^07K=m@grV(jP<<$L@!;@hbZmh9Q3(j%+ESsUqyd5ea8<3E3Bh5Pd8%CsKY=jhl}1N=8998&7*DtI+V%@Om71$qZynnE=2&SIK(BvqUCHS1b{#x9~YVOQeZn&vciF02;JJ6u7`8V&HhJzkBVD?<m)9KgxOh&kJx%B>SS_a|^G;JWsZ0iOkwqp7XIp2qa*Z$W<=?UeJTFmx$CJ`>199ga|)Ic~IOEu{z`T1k5GUDEBe{B@%!VERm+H`=G1m24jyBmIy)#OQZ&1cLP2-*%6U=HsIxCR=9;F(g84ejDZa9G<ky;`8gxhFo$DxlmPcUmp|J)z>+{6w@ZXT^j;zjOt_;oncJ-uR(R&wKA`JgQ*i(omq-BLzC>zxd+e#JlAS&5SlRJVLePl`L+366xaty7fJ~Q&HCDNOp*V>0OC%cSGM`<PY;foH96I2KE-(8!igy%3&Ru`{+@D@Sw}|urYDs8w;<qWN^?3~&`jhhJFKTT=f97fO<AAhvvS$zjY4Ig70oq$4AO*H0I;%b35zw2mL`qV2bz%nX+8ofFB@z(VSfU+lt7MO%_-w3tA7v*{@m?ZaNB9%TUQQ-uuyd`oaw3@<bo}n{D0`su<BdN!hw!bFSs5EHEt~TL_-vdU1R+bu%g&lZV6clilwFSy27%8dW{q=-z@uc7w%8Vx11>rzD8Mffs#8yAy4kqfvq#yqd>rkj!ef*eC}gAEV+?<l%kpCz6E!A0&r$rzLO!-}jvC?~?=lA=jxh%#-!U&c`{Ee;)X(Np+1A_~9r2C5Aa5Yn<51^lOm}be9EotvoCAx@dd+9LUA@e<1UIqwiDz?4pEO+(O-1L57z)1g<z|&boihLmn6uIB`OM4AJ~$Q0vxPL_E)#*g_Yz?Q<mSuFKIXDX?IcB>aK6m!x5SKR3(1jZnG~R?utc&UiQ6W5z-{KnY|n?7;9Op2My2Lh5QLhnC^a<M5D1>^W$yKyvX_}PnoDH;(&+a1IGNB5zRXN2)HsF9$a?t26oFPL4SP5d<gJ)V1*WG!AU35@?ukPkGdud%QNxMjwqIuUl|0&RK?!01sSV6Ms#9s=^A4LYH{*(6Pc@KdUm^};)l1H}B4<+@?VfG4fw;=FhD03KVU1rn(=~>AF+rGnnj$<mUxM}(M*PvPIbeCS<7am&36GHv7{k3uDjbjMRF87((mSWS)#3urt3#MO-C*eS2Iigs&@r^Dm#oRv_PQNIJ1#XQV%hU$XtP|<%g{=mH$$Ksdl_0ih{50*jnlmg3apmKEt~_L10=F$PDJ5m5<JhzD8n>Up!-Qhv}vZnSoHF<-%~k$BLkw6>|E}%i8MN#JGaP0xe^+Ld^-))gS$i;m%9y$-s0J;szmhiGp>RNz5Hxe`X|iK=;dc6g{C=>xzL;lBhe(34yLYpc2@LP^(?<eQFKOVHrhSL@G>;2juFj3g(RAZ939a>SyrNnd?CFY?PI#fUWlPy2<6F%6oEb>L#UfxjwbbN19a*vk;Zh72oF>PrI({wEkb?E){@{RS-AZI!@4iAqAUloHLhc8jvcxVjvCxtkwo;0q)Ccx_f7&6x+H|6D+#t-X_8{+6ul&kN>3+-_&SNhgC+r8D!gX)JbemqIbD)A@*}=J=^aw4H{h7mC23!yjC?jg1;!C=yzV(hYUmhg1ICU-KvQ@rTD{LNMJttJ6Z&=|81sy=h`<wKiXbXk6VUF88lQV?+hFR3ED=Q}AJEZxjdTyuBX0uQHU)3c!e&Ku8QO@C1F~_UYnf2IgFv%&SobAwQVYjzI7~UxOVG-x#8+1AN4pdSp;`t7L&O(?m@Qp?_ND!CrfX=I|0<w;DemQGvxJQ=KP#7<qn^n*hdCkOPORJf%8C6bU{VTlgW0Hd=fUaHv++?roAew*Tt1`3cAJU7G>?Js9*+r3n+KsR6c0u|j$3;6xMUtpMr<EM9)<uE565$94EL@mo0p^ZTxR1Gw!h_<o>2wncsAPIj)N!Hqio~mDbN<CB6!Nv>a@WFYj`@URPcoPMxKeHCR2?Rc6i{IoK;wYiBP(oFE<+>v}zw#CPB1{Nt->)JzGQkjmhDH0v-@bI29g<$}rEyo$k3@ZZ@*?3e2_0c9wj}Stp_>^UT~tv+Ym$lCw|9KKCervc1d+UEw8XpU~%sOw^mgLiKddT!c|_f(0gRSQHU;zT|BD_T2wG2=s_JBCRX|5C;o3-OJ6sV(UgW+;X$M(MnKr%1thPpAi-4{#6kSX8nm4;%h}XplLKBXD$h?`?9moBfb6qrDtFM@`?~#v|+f=(X(?#OcCZ|YbeBKaBPVBvIRn@Yzb%UbwE1?<r9W^hgk)F*;(m0Tu}Ccy;1G4Cx=+U>z0L0&iInEVdZi5gzzXDkb5@yva_!boiG!G3V9r`G|3?-eZir4x(2s79Ch_@z^npCLO6Z-*_Z0han4zv__Iezh&OREs^4&8%gs8iaH4NE6ynYn`h*M^t1dnJidB0mGU|-Y>A{cz8HBw2tXi@RM$lh|AiyC*k?)sLLxr==GQ0^gbBhGx`!c%v&YtcZXp3Y{<XmJnK<sFD)*koFb_xotG8LU?x$Nv~Do42!mLTNPvk$ZU=l1j?lJ!RK#I#&`Hnj2zLfyLxxkU)%0OjRpWrUE9pB)L9R}eB3z@@95?cAJD7tfC3643+U(SYe^5T2l;M3_+Hx-f&(oUkBrESH=$*2D$?iLfIFcB$Eix&8AD$Rm<~J<H3?q#WcjvoAD$%vzdAcGe>dx-aA<W)&C|QB<A~F=*D8mwo0Ux68P^?9=gYA1Dd!KnDyTI#Jp|E-@Qi?vOG<^rC{AuSHSuLDUdp6m67yj770>HK9GVylflP%ga6vNXUaE3_>t5hG66JvJbu71Alqhrx)DcG3384D{I`O1>7OarpfrpR*DY@M|_sXvXe{VFpgbT_L;cYk>RCfHD2@vBpfH#(Xum$BYrL~FOzbUVN;`YwCuR2n5dCfqDeCzKNw9mVW(Cl1SOm$6gB%x7y^2im3=7gZu4bjjb&d});QDbvW(lu<;0^<jp7cvN=DTz>8fT2Ya~XuiK@Mlj_9{!K#GzunCTeEvFn&<7xmJz8Vu@S#6NUMjY#fv6=7t3d6`z`qQj9z>8Pp0@q{|EIj(y}voYPXqB#-i(n+XIP$$-I*VVYugib{PPiGg-9yg}5vE6+}bOv5@g2rp5fjV5ZshjPisMHl7&Uc#9&N1$nmz7{bNm0mCGJuqoAXHEaEP~>hGFB~j+421yrg}YR5JtF0Sx^I}GFB~j5O*I$c9axV4m?VtD_Hth)$GfjWY)Q*u?L}g&F8>NO?r=Iiy%C%mz#Zg)3Za;G=Us%Dqh>uaS3nDtfJ`pQnT9k>p);m6-P7epu*lDg-}5|YIcsZD%?bEGPuXHsKdjmO*A{UppACVE_NM_dyBe~+0mR4+(ly;o((J46lGS`tf|QFU9{$Q$%aQ35J6c2R;51R23>0QW!tV>ga8}Wmz$L+SY4o#b48BbVPlxS-0UL(;<4A$neFyZo@mK2P+mx5!UG=-PB?46-0W*Xmd|O9#^D~b4Mo{z4MSyr4b)$w5l|u_G|GWRCX6vjC=;p4O+ef`lmbmIF;GZY)zNh5<T7AL%tS!i#z5%EMw1KM0nY{@gR;ShKHF%#!d9KV+^kFh+NcrkU2}tQw?)~A29i7*Z3EULZ4Pt|!)DDM9#?4-(q5Yk*E0qAew&IMl)V+RGM{Xhn03<)RNl6?V#cM(5$w7%ns5_eY9>{zGGgR8j0A1~DGG|#)yy^wvzMFIN09>LPK{tX%q}+@^zoe$k4WQl&qni4<;K@2SGn^ep4oGV-m2M$h(3=f8Y_Ym#s>AzjzwUO+%7pA`myM}MK3wyQoGV*Y&Nj8mz`0C)l3)_I!pwMC?Y)8mCiPSHt{;>?Qmt%IEIZE4D&P7fk=jFO?~f*v~jz~Np<|}ehA_PQ*FS&5osvIIk1Xt_vL3F_Ojbft?Oyri3PK!3*<N{n=k|zp{s3X44wB|J1g@BW{Td>Ff)`Vw61z~4ATN+0j=w4JZtvaG2Au#M44yvpH<)ZUAr`A)DUNXQM9k~>I%<@8)`JLm!ti*s5Q*~PyjAyj=@-9ztFWcUJ*j69ScTe(4s-Q2QV~`g6#7g1`kL$)=HrCki8tO#z+aTN?KwtmW|InK`$9b*KE3M&Wf_BO&*SLIGSaa(KWl`<g?8$MdMN;YjCsAs*i!-vX6;6aeTmd=7XErK7QM~>e)6ZABqZW{#MV*GvqBrJJSt>e|#h=ANw3A{Q8`5(UTAq@Ry{Or^sKD#-%aL8jHQtH53*7C23UaB~SvwUy?@UiU|5#e@R+N^_&RND<_7FCP9flrv{51t0P#5aB4bow|DV|q#ea2Tn`Tf`J5-Tmdnw~GsLim$zO(6AG4039kq1~ZF_%b$D=e};kZJE@ywgn##aQ^cwDg06<C~f={gpUh(gmALs?W8>~kehLeV>hcJ!j~3WRHyE#DHb#^cI}XS;SlY->&-{dF~zI&<~LF3%`|vTLqQv&}}rZUbepy(4JHEf_)n(-5>fT@z^CZ*ybq{$GMt;Vd_p7W1yV@ff!y1QdUj(7u%G6Mo3MZ23MS&M43AF3?<DAue>oB8}-DSmI5u&E}Kt*`UjPUSlBQ>M>E-)}xGQn@xJabh(E>s|@UOUxqfTjEe`>Z3H}u(OK?O=JdnR79}b5@-nnKbpFNAK62syGPD{Mrqqo8EIJjjMNiBk(oq`QFGVYl{AFm>7Dm_IxD6o&ywDdfKYMiYp^_M#zkL=HL&}8v(R~a>UL#(5Mzx3|uMr~<_KcAr$j8gh$`cf`!MNKbV-iZ=MaRzeA&V)f#u+U;JEK@-PG=8ile5`%t(bxEcFb%z>*FV*<7OLg2TjBig9wr1pr0NgetV}NLB~BvQFbB-)_8&(0sEkU21QNQX1k4Gc%d(N!WGr*>{vfDJY|fX0xL8q{=;rRvTo58&yMLcaJ9jWTF+;saR|qj1l~;#fdDK7D^>)xZ%0!)yT?!}#qM^V5Uf}U#HT_MrFlaKG)JANJ`qYV)F~sh5{H(3qm`#D9G8|=a3<6c<BIOa*>Qm_Dcc@fJG0#)2J>L?lCowLg6cO$h$miBRy*}W%083dNBw5dznLQ>7#0X;$6;lbA3Lq{8Ko<moe%k};MHAo*VnjNL0nQ+IxrXLG+gmq4b@M>u~TqVdx);B@oZthmeIJ3tUSv>jbU%B7}#n<KoTGV1B9?YZ7DuXDLR|op9y$;BH>J4I_6bVOJw5^PHf1gd+Efngk^_FfvB~6rtyBr8s~IIad_GjHYr3Vq9-}(_N8+Ub2OoBd*mF1PIDek_8F)4Hs*lI@f>9++te?IA;slkWS=fuVt*#Sk;buu?5bsZ2XhXzJ=xtg+o&ifA)J?OOXEYTa|&`AIThJtc8`E;ZkQ9su{@0IJ77L%HD{t6aneA=;3Sx%Ng}8hVmt~=j3>b^@+3pvCodx_4T=Jxo}{eRe~w|>(YTk)6S0D1>=Kwk1j$Rt%1vfiaY(iXAa8?XSaHajksaN0qArNMjI2V6dH7i8BM%?@K0drhDY4fbLlAz>myMN3B&AHx9IbYA>=?ZS3jZlToWVyLHFcmQb?QWxrj1@VI!lE-Hx;(o<y#}GZB1djQmR9D{nD{AA()!b|JkTBQpaj-!bKY)9-GF{cXdsS$I-rY>;tyS9>nmP9VLF#U|umT;SuryeTGj&s?!O$6|XzT?_^z|TXBUUx1(ccgB-;$kH!b?E%a>GvqNRTY<R|m7_P35os$|2RuckaX<&yCO^4l4eK;0rY<B0HGGG^2=75g!FFF<<i%x<eC>iB1GX<WXM1&y?g@$yL{m+CIoXiAd_~~A(_wXNkS|YP->fENkL<oS0C4xZh5)ojPO9bU-mWTnc|Ib^KwXr<sef|BqgpYFGkw~byiG1x6nINyVM5woxCBgVR0zB#Zi-KbJ5;2e$St4GG+_?k*X-g#AI^5GQ+gtw<sZw7@YlQCjFA;_oNlh;Nm;p8U+GG0EDE5h{OT<f<ar8<}7`q|~dRU(}<2IGIBxZd!IG*(|LC?eJuJroL)vbLat0jT@lh$7h9uD@_VK0FIP-TfEbUV^y70-vd>e%D=mk5CvwL};c2$qQ8>2_wjPoZ4h%C8M#eY7tQwJ-7KE`O8|mPi6<_4Kv(HK`>s;fIv)hZr7J6~L61h&B=P%muLqAL=j$X`tw#34^zd2K{b{3HQ0Hj~$=T5wa)kyF>&!ep8TrSt50x$9Bl((1CS5i%%=a`b)xK=#5i2uEi(Fa4r!l58@q!;c0e^zg(NyZq>RWTWOyCxDtV9(p-~%8sDEAD1X?&SHz479c9$Fq_4dN10J+Q1Rjr2Jc>c@)K$ptbfe|&OCX_1=73l+Cybd%@SrBvZ3@7OiAh5m^nvLhxAM>LYyWu)zeq!76M`O*IYyfRSOT+O?;Hej6-$Iczv&V|nXe_H$f{?@#_n{z_r-JkKwrOJLR$if+{K&&+<RCe69DWvsmF=4C1SD15Re#KB3crcBVcfAUK(aa!EuS8s@M`~s$!d+P8vWbG~2d}C2+mK$L?(uIf-k+e)QV$u(KLM1%@SJe^Q(H?FJ?}qQH{^MU_bjY@P=+6P}1xP9_kGCqq~;xk0l}4sVGlo~<E#KG~r`a+Hli8(2@qzCd8kYf9OW^bSIJ5Qa<cXxQ1qj)onr)Mc@AI9?VO-+hcrBB1AzCU%{ep#zj<mPnnI*`cYeifxafkl#)fgd3)!Ui&42jaD5`H%f4RT(QDL?X*!wr=xtEh?q_z1mCAIBFWPPAQV&36u3kT<*KH6o&DL4)&lfkEs?H2aRV+#owZA3LThqtw$6Z$Ocy}b=S#rq?Qw(H&K;VH2Mi3}f!V!7F9p+bx-n6jV+Nrd>#VC{cMO64`X>X6k4+A%$>7dh&#owTrxKtLwL}sg)NCBW-omCu@mNz*x#Dw_6P6Zc3jB<MI~z;HU~oxC*enBDesf>!*<yMNV&z1rYfjwon+Bl+Mgt-*8iEcO4Mkj?M(1i{PuH6RcJ$I@X&*mIZ$V7I+jpP0B+zH(L|aIcpyDROB~QU0QefCL4Y@a(O|okfPnv-j1nOZ#s6@U*8ho%lAQ9nF7zJ*6sn~~BDED;rHzb-rlurf@Dme-Y1n-J1kwlT5cwmvM+TV16cuSzB+9krE!EuR5lTY6J!;0b@RhPvM+97t(4zol8v29|Q>LMA?ze!VTyrZCw+9kr!OTJ8uEB_`6eY(`?gENLhTM}5sB}sTpwKm$j%{u!W1Sz3hGAW}p6KNB~j;W|3KtyVFYV~;9mWds$M7_x?PCg>pr>n>YA~$5%XboMvSt1M_oC&mRgUttIu&aa2q2sZDreN6F`Y2D((SOIq&QxA5W(71RXG9Ch1wnA~rqFoo*6RxSQLTGpEJq};-m<y?yywvpvAy7Eto9gF5QGrHP;DX5QeG;?g(D5CxA@Gp$$%d#Xo=W98!r)~qQil@=EO5d&OTv6*?8ImR9+%hHKb8uwvgCh)1Vck*R1g@IH0%AL=c^uO4#!m9uuJu9xyG#10q^(iP)ok9*XBu`+_q@W3qQ9s8xB1B#Ik6(f!6HGRp&Ywr4h3nX^G%J#3pkPXU`ewK4q1TkwPomnTXd@C^JN*z(CVP{#*Te<gPbz$`5jBd5bJ3zLG5b5jpH*?K0JmSJ+JH)>$loeRt*G7nUvXP)o?F@YLAGsDAyc^GqJwr3M-9`;m1m>nT!W>{+E%fsSBWauN(6~uN};FpK}j)MyvJ`#lNO+b#8Uk+9UTNZ`BO$_047Dq6KUlvxMV}WfltPpi^QxAJ+8Y?U`vLeMQ#IoEqBp#DiLvV@Jq4G6Q5|H1HM6Nml*alc~TMgU0gR5a|44s<^KttR-qIkL**$jFM^EPY4FaP>TgPbF3*@!%eImYdX>=U}`mw<8QEu29*nY-&@#~@t+xnjrh%N>gZO7pU#JI>G3w}9v|reGWdAyqJhq9Nr5-Kl7E24jE6%B<&{`!kNJl^jjl;JCM!gMD<F@0sL)kuoP1p|U=E9Mem}s!Bemz_^hLH#rS@rluNpzY{3O!I=oh$XgGS$^%MA!?sCEcR}oah>>%W0aJ-G6tRBk>S5dBWH?ftjOgnVGO%tZ#9cCHxy?Je$b?Z$=2sM(KSdgcuopZP!D8vIhutMRa$oWivEsDLtaCEksR(9a(pAIGieDb4MUhH~0jm^}538PkhC(5&Px4k;Sqs}`NO>ukRK8lep2VYff_VdB7{a8x9Ck$ri!egy@{@zbmxQ7$lrYR3$jicJRWlHdx|+FtPFD;&v+Zt&9cLgDPDQ9;z<WV&F|3OHA_BGF5wUZ^#Gos#`x1|3cQovHuI6so$zq7&81~)`lhQ>hHN~SHpcLT)(XTS&{FB;<D#%cXT5Sh+5HC0R_%035?k*h-JIk?w=GyVFV{S&E^eP<<yCQ;8KVn+z{wU?;VI}Gn!_L<Jj9@<{HZT@Tlo6AciPhSFnb^mIwpJP<c0a?2{)jiKJz&Qbo-l7JHv52iquD!^hKU`Y;x$`1N{NOooE;2M8<%L(h{vo0HXTR^jD(?Zf5IAgcSJti!Tk9h)TAu&g%9SB0F|&+MzT&M9^Gecvu7sBkO!ooER&=n$eXaeOVXj?x5cpYp+wfCWAqhK9TTOHbPy<->Q55(VJF&_!XBOIfT;r=27-^4x57>@l#V)nZ9uCdYop&fn**4d&WVnn&M(L{|3phVp~31)z<y`U9%QFeQMpaa*XvK3QM<=E_7T>fWw4JVh|a7H?W~vTo44mPG|{<Hf;2&2{xypLs6VS<--9SJ0&1F%Hr~>o6zrqrM5$KnF#$@blb3)|f!&p1o=ce;9JD9P4AsU0PdO`~Xv#2Spj?NptCYY?#tv2nDmtjl{vxWr?CZ-O_hg09`Fh#cEaH-iB0#1c`MRSZOrR1dQ>lXbV^jya-l_W0KKpYfN^J&2l@VB0#rFEiync{CRamIepVaI7LxK8mRADwoZ6Fe%Het{(0_i7x*;hqpv?X6>5dE_Z{bP@Ji?o^{2WhZBMlH&m-JM9n?Qv|fK5Ku!C+h^gl7A?6{s{Um&`@6LRp!mr4ds8;9i<f1n*!riT%vNR#>cg#pBogj#GuDg0$x+lU_=x(YQ(kG7BGLPVOYe=<y@6S>Hh9a67*JRVw3&JyuQFvHc0j;Eo*{s#=vg&C0><6(m>Wi6TBIr*>!q9d3XK!7RLt%u>s49HXuK+QI`?6Be&7O*lqGQ>Ta$(S;)HVvaT;5JoX@7ZOyhs8&UTG9SP`P?WJ7>*ReSP4Yf(=;MrtwtZYi7_s2+VHd@^lZ_`jh&^q3=0h>*j*|L{-%>ru~4RpW3XoB>Zy}awAUo4)95LZ@m=OM@&*~_}BD^vEeu5V9cUDfMY2|;TmNDPbzG&P>}5#QU()Wp|qBwfvG$9ns!dHo2U##$Qv*4oRss^Y1!qy6Uq{@FUdb(Z_`ty$p~lL-<yCXHIp2^Ox|?j1!Czc+zcz{F~pEKjfyZz6E)K>`tClLV;-yOb+FfaRHr))sr>iBhTjE5RsVVJbabFjWL9t-JblKXVit_V;Hb^Et#Ch}xMZ$~Tz}K$*>&lFSvMQYUBsjC*8iCJ6JJ!7d{+t4G<pV~#?oMRU(^FE`;i_fg}{<M|w=2&{Pm@k$)E#&7R=PK66fBiJY1`WgM)>f?7;?1(U%2Ly$Vp;k0r_AI)rb)Bn67F6So^A*4f>~gKfBSn)S*q%HLq!8?Ktte9!7TsNhZ71EGuf5)c4F#4>g8LlvOSd8_drTa=Qc%j#5?!{puHWzgm~=dqsE@``Z6-+A`xwCG_-J_S9fT?ex)F?zAFklX7?QllkUBZ@*rN}vzOuK^hoOSD53hU7Y)eEv58m@g&_Cf55?P<HU*0Dn@92{OXz(dbAl@sU$P@UqIvUJ95eGK!v*rd@Yl+~R(<u4!L${n@e!z)PTZ&U1xsDBVb*?*zmV@kjKu#Ym8uQPRs#D_yu1W$_1e_8iqMU2BZUf+^UF%o?6B>hS2D&{BOKC*Mnz?iV_aoLvMrQ}vA^&1qKai_4^r|vITHIxV<bn(JSsRzyN{`zWfu=5sTD|-dt=f>q`viqwn)MZWADrNljfL%bXw3%BVm@D-|B)DQt=Yf^8?FotSp^_pSB0KT4csU2rls&4IgL7PcO<gUFS-o`XWS+@Qbuj!$gL^6oLOAB`s=tz_@0EK+g5(*)$c*Bja_>6<<|F+vbwYQXw<S?%(0)#u0Cx21P`C6_~Vyd)l+qbf&J2}lBT<(o-Fqd67bUvxg&phsa2F_=m~DWdkj$4@n~@B*_xm-;GU^oX7#ao$Jd4Q89#rCRg_4!VOD4V!x0LNj<0U5ei5sm)|>+*Y(48lz|ngWoJiJMJsSDE$Mw^>r`DYI>~EGS>?_bDShh819jUHFcftFKyYYF6>qq>L0q+*XAe5^M*gqH@t$H>mhW5cbhBal%=h(!hRpkc92ylMm(yHIy>Z<kg8IAep)q{9xRbc@!sno|l6EPVfk(dGwR!prA#Zl`eRkhBhqf}l@*z6j!xeK$ieL(|UQ3--cfFJ^qF{lOtJ3<Eux}g`OATt?cAlw8wV7x&A$H*Qg29M?{+8%>z=XzAe1l(_2TJ?dQc4j45&}vzm^o;F6y5#kliPtlE;6`wOf_S{IXH-ZqWFTxEms^#I=a8oJ;yCE9#=#$CG^9oX=SYYmr`P<V*Oh=xSRtqhD?Z=On16O~gbrkBExUTSb2LS)b2yb7pT>I-0U<b4kt5N%rRBI^g+4$7aih>CwQ?eX4S?k060DLOhY?~wVO*ll>jz9g{yPjdWQG}BjsyM^76AT6$FjDB*L>g{zV5cxCP?7~@|xj|xgJ#Vd0p%0@pFZfMT{UE^|gfuDwW0MS@qt;06mfsvw=1{Y6bOax(Kc_pF7NlKmZkrXmoT$>rB-VYm5jOalRS`f0`O@7>rA`Y8%Kt(-M$pMG87tSyuy&mq<e<lAx_EGEnHutfA2}(S((&Y&q7Wj~rB!^-cJ*;aJ;1a=^Z}914(64g*Q$91e;-Iik}0-ntwKz+%n;_+rip(y%!Z3dp8caIbZuSd~+I{GAh(uXENAU<clubusWxBLN-BORhe^^))f75x_C_?sC?S%t_7R-#bSFVt0~)B3+UJaFz|Zx{C=Ag_2O5N`^gG$rDtRv#wd~@Sn`B!%c4bEbc@V^_3(WGIn;<7Mm}(`bhTLmS_(DmXJ|y-TuhStg0MM3QAn3!0zi5K`F%)khMxdQ7Xj|6-fzXLXt>Ap-0t{uO8x@O6b&GURCD$^QBc@ZNDmr%6Vzk*Zyx;rm!U_6)IJ0r+S>-WmR9x-GS_7Rb{9%0zt?a6wcF73LxKmSW->{l~>ceufU}R+7?R$mVES?)AR|jbUGn6oNrHHJ&2&N?a)je^{b>ia%4Y?R-bOlvk}?HGGMkUFQ59bkUf*jram-sAVx6$3QDSEDDpzlG+G|5eRc%?#8eT$&@P#ZU*;~RkKz;Ms`BMhTqrdOwG?L7DU}<SQ&6oh>)Ti7;HAM}m-Q8avw0a14!w1&A~HW!tJdEwiH7?>Cu@mN?~9g1T?RN`ksd)ViP{A`e0xb)-yE|fya!B6B7i?zf6pFLKXty&+?T-YS;&$gw3gX4u0E;RB_efF{Cq}{flLEQnI+;)9`YYIvP7aU6fKGD0r68B?Q^qBWP*pdmk9NRf+fLy9cW2N-z&Hz6qz$w-FD)S&Zuc!Ir9LwT9-uDP-GiCk3J{9Bmf9l63ulBezeYv+zbmiGjGAl3QoO548-px;%IX@T7tL)>NjtWBa+ej|0OW{#O{(nNvb8%#j1V0McRbP5>TWLu|DoD36T6>5`nyofEwV_u~4;o9fznj`ac2<Pu=QJXr?MX#f+f}VtWkj?DWi}#{W;^Z%M$zuVr_|QHTuyW|s)c{4bF<L%sf9($wp54K%GS2}P023{TYIAkin~mITQCJ#8_-U3#5SIS0lGhC1iI^>Il^-)XQUw9Wl20qcucOTxiryd@G4Kz22%ZHF<ClZiS^ov#GAZn-2FD5NV<-LGh$`&kkOfcp}0<aJ>$;1U77fKU5XPj{Bc#3F<|Q8L22SsUwt5UG9N+mbK@WzzeC%aW))z!5-{Y7S(ob4+0Fb5Kpp|4AV&3GQLa93W4%Bow?^vqTIa-T6|g-w4w^ABg};pCbVZnR9>~!xEVwA+kiM&oeCvE<xr2%yUYghFlT~E~hRL10i5eaNl}K1PC;9BDXqe^m*AOF_ASlZ@6!_I;lng_bUQ;$|Q9BCxJGVB~jL>k0DZ^ZJ|UaR9~#x-{1)j?qyT)yCdg-;mO>$4=o7+3}e20N<Tg$kA#72$z(_FfdfIz$#oO(k&xP>9!nR*QKwKJ+NU7eKf4mu9Yo=B#n6Fj2-S|bQ))Xs`!9(sL<c~_Q)k~evLvX<{A?4^zU;pQq;DZz64;-rA(S$IK49^crJ9Q1|I`GTj%iS>R_CeDJdDux8Z_3IQ~l1UYd(f(+E>e#1gNPj39p^V9R#JRmq?kVI^2G~q)IBC`9ylhq-r{7rdkpL(hw7ss+NSosXClTpi|Rus$<MCP`Yf!tWyX(6up(I-vsaiz^FmhU2sTWn_Cjy6|3!8!%&zA2Zf4RSE}~jMIuAYfp#V*aw16xmL>P;(wPuZHC=yq449{yiSns41EjHMCa9=AmsQ$6mL)O!_)c|^>K%f?xuk*2lt$@PkCoB@{S-^0VW}JI(f~A-MuHBs=Xy^IL}@nEfBOfTsG34AqpIwXrZhlwPXy$tXx8_D?_wQQ4`_l5>q`RFsEDAQ(nQ5FBI^4`mINcuO-+sJ&J{ctyd(@%?Wkr{XX{=@^&5rQo<%mh|AZLg0XztCO;XRfWJ1KA>c+pL4hFrHOQP=lxofc!{Q2B#K{1LrK%66nhAN;$O%0<uN`Dzuy>rcT6?YJXr>MD5wWB3<jOu&^>Qn-_nC`KPoh+l5Q58)rOJbr!sf0XP-Su}hrckE}kG>j6Kao1Z^`y~P<VYLfzgJr~{oS>66{@{;M#u^c+Nr_FZwgLSbB*da%a=`2<vuJx@{z0y{ClR35GwgZrDbvgK$w~e)t!g{21<_lD9GzRiswP4CDFfBvjhUE-4o>wC`QACj|j|AQ-IrPOM(Hfrw{}mLXjJKsg#uAG8}oTfQ3JbM1$l&&OMuQ>XD!-=)PY8{b`hfVs0wP>romggj3cI;J~FstrV0^E5$!L=kX)-&)*3O9utI19t28@+&=&JoCp5NroLPi4+VIdf6}RsG&YY~sXIZzBcMcraLBU;soTx)L{&eY;3CD6VDKQy5|N%7@>I0DkWTRgxSwYL=g+eVRQtd%4YV%=b=vg%-LEi!<V?t;G9i!3R0IDzMx9DIN~M5bNdRQH>GwM)w;@wI8^eqUDs!#V9a0&RDQ0M=472`RKTvTEgag#Um>WwypOJx9BW26~K~yX?^uJ$WbRI<dizQ0}QB)Qy74i<^ZI+aqhQGVmM8{H_JCtYZbfRh-D<MNvR7znL0CiavWfxctMKX0jA6Wxr<5>}&VjCbI$2I{>VS`G23Dqn>h@q+<H!Xj63_}OT94QQ4qgoQ+jjHDjoVsBDd<Ay=u@iE-O>OD`3|O0DSAY%N(6+cEr8~IlyCk4W16t`@nW}<zpiBw}qoqWx(X*77N>LRG!w{$o)R)Q;=tx2$#W_H(g%iU5r1px{5@3YKvaU-X3(KFFe}2uQ0aM#N5>-(7Wl|-G$!u@wJ<n40+aja6=r<n$0tZe8c$j)gpx#Lt3Oe6qSYHg1mqk_hRbCdw<x&!Co4hQl{L0IsW^wRk5<prq85|aw&B5Oao9M~UC1T*j$#eh;<mFLCAeEUQLnefnsn7&RK?r60giz~*5K5s6Dd>BVmqir-SjYi15kkIH2xL&?<xsQ8!3j@vteF5L5Jr_#d7q--YNGJYlrXCC2|Hj6!U24`FpBsh4Tw4iTP8ZTFNLb2u}J^BYD+-+5GSI@A2;==4YTB}M-^#4kwAV>5}9$+;CCj9ZcK?1NYIO-<cTQCoJ!l@w|&RUo@O=ECdqf^5#Yj%Xo3QbSbgx1|1s!FjX>LtT=L}c3joPd<0B%79E(9aHA8{Y9F75lzM=MBdHGZ2U%c*^Jo@k&n4XPkewpe{LwZIbFMFz7!^@sZBXk&^>c+lzo(U=<5(8Wwmw>J`5FwR7*(izf`su-?p~yFcArBOe&QsJFYA03%vM0$VJe`Sj?@#Q#O+xmOjBsbepN`g3(Xmd{fGX)A(=3-h`9}=nEs=HA?Xf}~12xp^m>{941GS*(VBmptG&PgQ^XpK6<#iY+GHQKQ>Xirpwe+P>75&kP8qjr4U}QQ8Y3x!c&*Dt!M3;KCd3<7@T&)XHS059+JhvnQz-XP>!)Wcg`ku3|1RhCS5`<a^lu-YvLJFk>GJHy9jJifh4p^8{^k_^Q26fiT16deDl`?_*p4w5UGY<j4K^dNWHW0Fv9V9-qp-|^MaUvrWs8jSx`jQ}kl2s5bRY6%(eK}O=#ROeERdKi$M50ts|Cu&bsB1zx!R<y>l;l%okO<PQsqaixK~_T5K!HH(U-`Wzq|luzD($JEMoH~n^4niAf)c0(ttJ2|sVV3!QbVa|HAnVgh??ppQ6J%Uqb-F^ecG|8t#xni@|+XnsQXl1z-2CNc+^puTTjK*VT|&!@%&68L7)s&FVXH5zinWFl6)H6|2EB%5Ol#xqqfomg2Ir#{%Yr&A4fmx9_g#IL>@Ososm2$$vVS-%p(c;vE*858`Hf!sv?D&iacFYhB^k24$?uI0mncI6dMD4#YU~uok6jIde3YyNcGqNzdExdnuPTIjKPUGaF;ev_nD3C3v)KnjZ>Smrwumiw$n4u=5o}Y=KSeQ=Wmo_QvoorX?@<xruXjFCQ9&IQ-<1C){F)Swniv{Vnl%L8F5cdjOufLKk}ob|I>GD*I+e?x?gd)*Z}y&E{XErZ@^GKGvxs+kkM6hV}cS0Mj4~$5RDb&&5hBmNaNB-&!#Xc=o@EWOf?zk#7gZ*cj?Ap8=nbEMVp|GGZPe?tZCzW2SMi^_VJk{3M@KMm$&Ie?pmQzl|f#^6m{{MDx3%E;x%<B5uiwEO&8x;E$remo51PACO&JL_%@-m=8=i>5tQOHGlXu;k&7}^GqH|8?WOntfurl;+aG!9Q+XEZD^d3&f@XpG&CF43a)e2(yAZX9sGsZJj`WhJFMV$}851xP3xaVt7#+4iFy8{*+p#df^DG=4#|Tgzut51eYdZLjTgZtHLK5hzwPZlQEJ2Hez09f7IlKvF=}-dvT<4NNwGtnQ<NKI^YxtlZg!w2z{@E07o|U*`D7d2N!vMtc;Rp~o<V`*Rgz`B+o3dZt^nq)c_892>BcBXhtxph|d_t8@gh_p(yuMHHyN636>q6?|Dji;f^1A$VclI);vR>!JAc5!&bvjBXR4Z{pJtt16!Q2V-L^>rp3tY#i;~HM-M8BJGwgE9}F!z@_eGKtTw5dKFox>L;oZ)pHztpKjKh9;g^z1Brq|c>R>0a!mPPGXN%IkSue7eTb^zWS`DJWs$VkmIb%<1S$q7Q*CsO7{Jb(y-(9)`IRI4OUzrukuC3LvSgf`q>-!uYQ6&^N5<ETzss?e{tSXZPM0p!A>Zd!L8NnjX__V1}A_+{#w7v$LikdKn_3-u));gOA+M8*={hVZN_>ZSM84KS#mQ!`$u!R_hG1F5Xq0_P%x9dwUk3=;U7ZnBLgs=z^~Q#hm8HAb~1*4hv5nggjWF)(8)%(o3Dbpv%}21{_d0;Ike9r;&irdJeF);J2f~yVku8DH2?H^ehkk`FoRhzTZHXNIg-HnKuQeI|kxSF-EVh7^uZ88rrlsEe23%41q(S*Z5*I=xLwF7^u-CM)c+tBZ2kAtbg8Qd+V69ha52pybKc)U3iQspwl{FxhW>9HOKVE(znJYO6JEzwdQDe({^z|=rUPA-V_9CctPNnPLTG{CaCtJcc~7i8K8qCn%{~#M#|`RUhq$h|8aF8E^Ydt|02(>G_RW<g+SgE&Epa0<g)=b8+FBLN7>*2jc&m;xj!EfR2B`HppQC)fJ+ZSkTgVqw-p8cN6>Zr5C^Z9g#=s)dMi2<Wb#8NSgI0o9Ey@9p#s!(X;YaZ4>f>|;-yVhb~|9ZUTA`g9$^DiisI6yGKUogRf?XB2oq2?4MQotFzbm|z+zKe*7V`OHs}ol)sg5p)800pAZHveXR7SW8Cj1b6Lm#i!ws;Aa8S07pT(w6FyIN`W{ehEN1(@dBB%~rXAG2bjzFoWXlhOOD|J*lVLBWT?g$u`A`zfS)KV6STFN4SlF*OjQY7jri@Ml!@n)jtlDM>~ysH%Hperac5bn*o#=Pf|1ND?;)2DYO2src{$WU{DlwQJeQw{_EJck1t%Ml=I<Up_OXZK*lR!8$-#F@t>P9O8wh%_hir8(=2-eX}o!H>c(2}4Ss4)WtUQJFdaDmTsF?luY4nG<&EB~kxSLlQ!yNvL$6q^LDIFLf#lJ^4~6D=<M)fQ)ax)Tx>&g9KFaM4Qhbm!HfaFeD2oswboNl4L_G@g8C&2mD5&%eMJen|=dudXB72z*SQS@T$D5>BBU3r8-5x{g4!tBTYe-`)vC3jzGZ0COXrK($0B#(^1F#aFmL&da0<XEERR=W<#C!QBVUUF&pM|&UDS34vI?67IkKG&FLt^&{c;t<VOu%#7zUtJ`LT+NkfO;9NDl1ngwdnOGlVA9ePnm{m$9cn$A*ClU`olRCc&z_fo|9h=YFC^nha|Qb#lm&)Y4;1d9y_NSlWuu7p4d1()bDph^R^=4M2Hq9Xy3$V82g*_4{jT9c4hXr9^ICDpvNsZ94~)`96bj<i1inOQ@pIUc`|QI|W9CvIrdKT3Z{Q`?lkBxd`Z(h{J4SC$0#yRsy}1>7Y8SkscQzMF7KIGkzUs<Q+@vFRb1ZKiff%-#tu32H;a65!qfmxO?AYg+Wqrp*35xFz7=Eu|&Wzh<xmKxBG2d3%<YM4iNc5_(I58gw=J5#hdhcS#7+7-^rSSQ6I9geB1!>3CGuwCEl4phWX%di0Lk8`5;gAYdO1;M0<5%l7N<Ag8w^tk1_S32(vC5&-4UC6UO1$w2N&COFgReZ3{Y<w@Y!5`wn1Y0x`fQout7OTychq9uUb=wCOR=I<eNl<l1lmRu4O=}Q56S`v(obn`UCIk2k8KPh-;Yf1Ep{3Xy7M)t;D!t{X`o*Ru%R<SG&Zy#tA`HlKzi07925&*YGmPFn9^mJ`63GH*2OTy}_yT|1>tZ7@Al}38DzAiOgP*OjxckJ9}Pq&u@q4Vh6J+`p^E?}Xq00cTjPIaN_jPTPu=f{>gcn;>tSc7m86S(pKmCz-@$WdRu<fW$$ML6=WO4FD73h$FfOCk_X{8?)HDDOQ<{x*rWB%mK{IMbO(a{uVel8~O=EeS=(?JYIwhm~z?)Uc-G;}Rj#tgnXeC_BgKs~$^Yg2*!m_0yOGg{C=BXqrRe7=YL>i9Y4L1fnmeFNv}udUkfsK83#o`UbrvP}f7RHYT@DY)<J@14}}IhnlnR$z1{<EX^6n{JJOg9v3ztWts#YFIW=PYm2xq{wxUru6C04t>;U^+ET<4XevyPOPGY9)BNi(i22(&P4p`Hl9<RTFacOInSqm>+&oxxP68|PJ$0Gs4gs3+mjrlHZ%O=_!2dbXvsxKVwRAflLB}4|nU(|!_*0tDwD(rPEu19*+R2uL_vQ8_@$>Qg`RhXe5=r2UyZ*erTAfe=g(hWykfsVUpr*`p$1soxnO%q8HV+LH05y?!Fi^W&60=3<(+Iga<35!#n>y36FH;bCE@!HG>z0J~jA&Y*jc*D~XG<J~dY?#uYQJ&z%o)L&=J%WqYH{t`g_eZ%`|b|1w9{*aZn!!<(HJohY>C-7t}F@AH@+l#7CNKP<jqh7TEfste7p0!I!8@+-T{Gcl}QRfQD0};3(bk*NjCO*f@8v<lmd?PTVj&$I_Q!xwC#xsBC}yj$GDo8$hWJffpViXK$xdNH8*^=g$8&@Ye@*425_MS*%FNgImy;W^sb%;O5{<^mbOcvy25n#WD?ML)0MXse%Vs#GH5D58g$vxS6SvR#Q`{@8KCKIiJ+)NGysky0%3^=Wl4w_PDAQL{g6LuDANW4gu3#U1ZZ_!672)8>+hge|8yXhiI{C3+&4cg0YY0}Kc9E)oCSRZL3?6UvLkjt*@*+3CQgvJBpEeHJ@(a<n2rD;0j?wL-q)tWbgWDI8+S{<`s)Bo0+j4XP|1#@?;3sDTibiyOJcT=hLjM0C1p^LBNdRUAy{7`HIP^!_4+4`Hnk*3l}M3F4P0Sd5)<GvG6Hy)YOZuYA^=E|fxgitVaUzamzl2bF_38{OCaC_P&V?3+^nNWU)Pt;fo`rc?R|d>O!XwJFF}!`6Yqf>q@&gxYF-`RwGfI405Juj6CDheQeR~{)RRK{@Ed*e*b)F0dWt}9JQ6vTZ1Q(|r8$x7N<=&<(Q*Iz^uzqnMrspq9(5#0+f#xn2z5W%w2x|LAAGsvN%IkyXap*=<vX3guoaJofF1xIpq|X5zX-Yn5UA%7pjUxMowz&moJgN_QR(QSxxBQUCxe<2Pn26`^i<xG06&4VBpgIM?(Qx<M}Q~l8Dkp2{Ahx#FB5|FObpHu6UdA(q4p7`t~|fDp1@7eB@u8fS=Xz#cjLck(nn(=ca3<Dt7axNR|Z!>mjvn@V^;73CQHOXm5kZZiQ<40Nf0wwkflofOy-%!rAl9U!e<tOpgoJAS=6{`^F|h{N#8!U;vmV#62M&x%8sy(=H-zmkh7wKGAq@U+I=jwW}nB>|6)s@6G3!|8@hBQ9H1lC=n)ueB7GPF_iQNM*B6+M2_mk<CjIr?C4r*7^*<lH1RR}LLs@3FM9$U+ItUKl$XpWC;%)}5c<d-S!Y@_g-v+$X_S{sJ_TRl8Vke0CEi2W5#}h%t7)a%TS{^t+!ij^C05s6Q9?JXhkzxJ&UrWM4K9i$=hHMFd><H&Tup}qQeQ**2X~`%l#ffUk-2BSzL7e%6(Erumapnp^eg+CW4QG0^E(2T%TN0pWR7Mk*cn^VEouna6d-r81$RbPk>cis%WjK`RU&dVm3Hl?YW=)4F$ebV*Ba;B(mWdM0GLbKpzv@dLrY_UKd6D|(nf6(b32uTe36v=jf;uJCge%<@rhT5I`F*CNvW7dIt?S6s`HBJTEH8JeVv96AdE2WCAy6vK6{f9IM)&N5!DY}T0g|J_(jfH5zp&O4cPR!M8-zh?j&MMIHII{Qz$r2SArYCNzAZu^7a;=O>04ydZ}KIg0PKkv{0Qj86+xZ2BH5=|MF%<wI*~(G>H)ZPMW$^%D#$n#v_6Q!Q?IByM>Ohf&HWYBjVCrhwpVPzF%XIpW6)JB2E)@!oIXJKu^&T)J243M#o)=e1oav5_Kmmo_X(06;sgw+dHUf*ILMyFwPy?Bs0~*fp;B>>FA+yvRAQi&#6+Moz{6QfqQ+1M?FINzktLxhp(26OB?8%#1Zo_W0JTubf!s?@)W$D~?uW_CpNhPnK=vf5pmABcH}j5!qrFX(CzS*b5k9XH^`u0{1h%dNVJL0r(_XC(fnQNj4^mWg(%~S2=|G-T2kLv$InbL3`lIah@k<#?f&eJciEfAK1TVZT32hj~-k6C3lune;*S9~9ibhp!2-KM>4Zs(aCYlG;#nH)I0+CWB6rz+AN`NQHmPE~;rfmuEPVSP(A6oxNG%BNvsWPhHD-(cp$_&7VGCchqg*rN80~r`?eyAz-OuI|WTdNEn$bQ~e12**5p5*&spCy6H&MMqD^{Ak>G8HX%?%0QbrU(^;__d)>S0a%FywapP!BuM2x|(+!sv=UV^`ql^=T*Vq>#K~%GP-y3a?i4TLu>s#`@h=&TUwT!u1hd<iBXNAgn*OS-DTRdKx|aak<}Q;=&OO>2!Gx=>`|IJ_&vrY0Tt@dG+k<{`P^#J?x$_HX*mq_?x29H9>~N!QQ)ed8$ZF|rA}O_wCT}1;HWM;9c{$VM*NQonua3a8=&h^o9D*2okQNF(bVzwJaYi5sfjWsnh@kCc+Pc6p!+wPiZ1r)&u8=I*C37}U(>3$x8<)_^X6~nWTWZZ*`Kk7I_;QauwBmvgXm~O`bGpBO~rYmNgD>pm5u(#?3V!0sSVU#W^;h*r_I@)C!44_*CxuD*hD1~n+i<b+IvlV-zLINRu`Vm%Vsvw**8MZycvO4a@RlL2XNLu+`BO&0bbvzL8Gxh0`H1h!<+6P(FYfU7n_Vx0@}DX)8jq5n@w9E%}WsHxV)^%DobVsw*ZYnAGxuk7k8|_+H@eJF@pFe18H3*lt33p3|?+BLHSY>bk)HG{0Q@saG>><HPs(M857f5rTa|k1Xm4B!Oam<^jwLlAeE~k<ACjYrd}e#CueJlE{~X^z6i4cNV%bQJu~#eqZ!w*b8`-}`ikfY=cXtHX@>eD%uoxYS${{=tU*y{2p`DHHTRp2$96(lOhY%E%wf!LULC*lu+j*3TE|}g^n3fq++(1ezBz`bfgXRdm{5YyxeZ2*>lT3KTcD)A1*+oPUli)g%A)IrJl{!BLbv)^h@&z{qO57&+b8DbP(|B=CCchqf;zsX0TOSCx_2xCf#^gfQhx~)SJiPJ{ry`X1pW?wSZGP0wmom^PTRToKql(LK{)r(<W0_35|DVG12iJ}L`8h>`u29@@QL1j^5)9ZcIrN%Ql|m`<MlgwCll{8Kmpxnf-Vn7SDu{EsrTkbZjTibg9GOzsGr+O`xxg0RZ0#}lD}mqDHVn&sq;V@d-;<VF3Ov#)BQ?+FV9)P>p9NoEq7-_PL3U}qz6&n{KoBBsX%=S3isah>>Udfl+^d8;<Q!X{8-absDPgmToUNPB9{O#-38rza78yCTqinA&26U>-XME&Mb&0k1@PD#_H?wa4(EaRsB8An#%-WSdfZUEt{b{6?#56;z-<vX?OSdAudcn1?4CpR<W1M!9V0<8#hb3ZCz0SkpVIqSt5d@_ANz4szIPYE;2pK?xud5E{jb{7{L)4sh~x?M73dWxkBRdB-f^d`k6?hoJp}yvfk)H4b4;uuPscfggN=#kyWu?0!LNDKSx8QFufY@D8ux@yKcSVTqF|kd+EBcHC*r6hK%U97dyM#5ZHxg1i7^4ZiGj+_G0@46(G*GVXKJPHa2x~S(iq;;;uxsx93%T%J~0Q#00k;O#RMJD0q;D;>|0D?qQ~FjB~Day0>pGsH;9Rf&e0T}_Kr~BPae%LHti&N5bA-7zp73jsjwh)?;!~GKt}VP-rkk&7n}Bhr2u>-Ufxvp`3D0o8cny}SxWa%#NPX+Tkoue(Yueq+K1U-1Ccto`DL*?2=b)S)t&Zd1OfdJ)TbZtzEd>Z>1a#@0D=%ybB^Z1)3J4NWFOLWK%S#7&6_4?y$XKfdPz(`jzS3_!l-}i_S79iyYO^AgMM=))X;tEQ1IARfUdnz^!pfL12q&xckSs&K%E*REEa|;dQtPIV}~z)lA^CJ3`KnVQPBJAEt(y#4Lr9M4xWb!_~T9CsE!|w8hFCd&kcp6Mzd&o_l}2g&}kMgd#VGF_syjNAB>2YD0T#T$|$<R(-jf4aiK`*VG!Fk&yk-96ak$_B!Xja|K2RhhLPYU-DoOL$4;Bl)3MW+K$XZc72KDNG?WF3bo4+<WS|Sxk%{_N@~;Ba?|N;H392$V!0)u?U;TR@Q_YS<Z8OU5@{?={)LWi^-G2IT?KuK<3g$rR{hZ)dem1X1903AnGn*RJS;|BWXF0WJggFhQiF3lcxj9j#H2<nWeeDZ@Ul~mFG<g!js7c_Vs3ZaTf_$kI6}iMDL-Rzx-j)>bU=#AF$>@4+GJ^fdh@WOzf;zdxWLVWp*1oeT+0cP(UfjEj4b<zNjINSq_dNMg2&_Y;H0kJLsLkUM4+_cd=F^$SRqF4L?0}kr-s4Ddbk{fqJQ0_YeR4ImDVl8Lm-QRvN2T5C5qla6L4Vd)pYAl!WwLDQP<I~SC7x7pVJ8)ye5-^0aWDqbd}&RO#HJqE+<?0C5I9p%g-SEHZ7+?g49o%vz<BTIc1b!)OQm<uMQ!77RM<=SdPKq=zH|dbJRL?Tk2(w<6jc+c<6UCFy?v?F$KzKAzXs|&$w2Hhe-xsA^6aDcK%M)EUMtT)Z*pV=co{U|J*dn98agsT18*jQp6P6^K^^-@fzyCF>FkKqaUhA<X(lS`XLk|m9}M&S5^mpL@V^0?(23;'
_HIST_VALID_B85 = 'c-n>3!EVAZ3<lu)q`e86lGusur$H5Pm^SD{Vb4D&tOntda;Pfu$xeT#JzbC6>(73;9H!0pI8B@5Fnt~N+hMnxZ+GK(-f!FDZaiJi!*u!lJ+wdG-*O82+k}roQLn;!39;UW3>b<E=m?;!0%uW*6R{)}*+h0CF{&sX5vA536ZkBWL{h=C$lOGHBm}jMbV*RsHgY4`lM;0+C3Xpc63aCwPcVrRSR^+AQU&&kKw%}8EV3&Svrea2vM5fZs1q)hEZ!a`-9;3PD!3!KZo_iL^4z2WvQ!a$57HiFp`wc;QmbK^Vsteq=?Nf<6vT;)I$g3z{Wfwv%w&NAc@b*~3l!K1WF0<Ppio|Pu|Ofe=pt)^Lq|Y;W!NHx^g>vq;3KVx@;%Y}dXRq*bSjL!5>4ce(73oP^q7}Hj|NkxfSwH%mS?)n1RnPpl19$bC0Sntu|yO3r)^avK9PkgmT4-E1SvW?fi4Leir_lQ>%^i>=n2=4nOL|fFA%JR*Q-dkO8q3Z<T_<)r#}Q}PA>'

def historical_embedded_records():
    dec=lambda b: pd.read_csv(io.BytesIO(zlib.decompress(base64.b85decode(b.encode('ascii')))))
    return dec(_HIST_TRAIN_B85),dec(_HIST_VALID_B85)

In [ ]:
def read_original_tensorboard(run_candidates):
    try:
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    except ImportError:
        return None
    for run in run_candidates:
        td=run/'tsb'
        if not (td/'train').is_dir() or not (td/'valid').is_dir():continue
        result={}
        for split in ['train','valid']:
            d=td/split
            # Multiple event files may exist in one folder: EventAccumulator handles discovery.
            try:
                ea=EventAccumulator(str(d),size_guidance={'scalars':0,'tensors':0}); ea.Reload()
                tags=ea.Tags()
                scalar_tags=tags.get('scalars',[])
                match=next((t for t in scalar_tags if t=='loss'),None)
                if match is None:match=next((t for t in scalar_tags if t.endswith('/loss') or t=='loss'),None)
                if match is None:
                    print(f'TensorBoard {split} scalar tags:',scalar_tags[:20]);break
                events=ea.Scalars(match)
                result[split]=pd.DataFrame({'num_updates':[int(x.step) for x in events],'loss':[float(x.value) for x in events]})
            except Exception as e:
                print('TensorBoard read issue',d,str(e)[:140]); break
        if len(result)==2 and all(len(v)>1 for v in result.values()):return (*[result[s] for s in ['train','valid']],f'TensorBoard {td}')
    return None

def read_original_h200_logs(run_candidates):
    log_candidates=[]
    for run in run_candidates:
        log_candidates+=list(run.glob('train-*.log'))
    # uni-core logs are segmented by SLURM run: include all available; do not assume logs are complete.
    if not log_candidates:return None
    rows={'train':[],'valid':[]}
    for path in log_candidates:
        with path.open(errors='replace') as f:
            for ln in f:
                split='train' if ' | train_inner | ' in ln else ('valid' if ' | valid | ' in ln else None)
                if not split:continue
                m=re.search(r'num_updates=(\d+)',ln)
                l=re.search(r'(?:^|\s)loss=([0-9.eE+-]+)',ln) if split=='train' else re.search(r'\| loss ([0-9.eE+-]+) \|',ln)
                if m and l:rows[split].append((int(m.group(1)),float(l.group(1))))
    if not all(rows.values()):return None
    frames=[pd.DataFrame(rows[k],columns=['num_updates','loss']).sort_values('num_updates').drop_duplicates('num_updates',keep='last') for k in ['train','valid']]
    return (*frames,f'raw H200 logs ({len(log_candidates)} files)')

RUN_CANDIDATES=[PROJECT_BASE/RUN_REL,HOME_BASE/'heart-molecular-pretraining-v2'/RUN_REL,HOME_BASE/'heart-molecular-pretraining'/RUN_REL,SCRATCH_BASE/'heart-molecular-pretraining-v2'/RUN_REL]
history=read_original_tensorboard(RUN_CANDIDATES) or read_original_h200_logs(RUN_CANDIDATES)
if history:
    train_hist,valid_hist,hist_source=history
else:
    train_hist,valid_hist=historical_embedded_records()
    hist_source='Embedded 510k–1M metrics extracted from supplied original train-1013205.log'
train_hist=train_hist.dropna(subset=['num_updates','loss']).sort_values('num_updates').drop_duplicates('num_updates',keep='last')
valid_hist=valid_hist.dropna(subset=['num_updates','loss']).sort_values('num_updates').drop_duplicates('num_updates',keep='last')
print('Historical source:',hist_source)
print('Train:',len(train_hist),'points from',train_hist.num_updates.min(),'to',train_hist.num_updates.max())
print('Valid:',len(valid_hist),'points from',valid_hist.num_updates.min(),'to',valid_hist.num_updates.max())
assert len(train_hist)>1 and len(valid_hist)>1,'Historical loss series missing.'
history_dir=RESULTS/'training_history';history_dir.mkdir(parents=True,exist_ok=True)
train_hist.to_csv(history_dir/'h200_training_history.csv',index=False)
valid_hist.to_csv(history_dir/'h200_validation_history.csv',index=False)
# Smooth logged minibatches purely for visualization. No smoothing of validation.
fig,ax=plt.subplots(figsize=(11,4.8))
window=max(1,len(train_hist)//200)
ax.plot(train_hist.num_updates,train_hist.loss.rolling(window,min_periods=1,center=True).mean(),label=f'Train (rolling mean of {window} logs)',linewidth=1.3)
ax.plot(valid_hist.num_updates,valid_hist.loss,label='Validation (original recorded points)',linewidth=2,marker='.',markersize=3)
ax.set(title='HEART H200 — Original training vs validation loss',xlabel='Optimizer updates',ylabel='Original recorded loss')
ax.legend();ax.grid(alpha=.2);fig.tight_layout()
fig.savefig(history_dir/'H200_train_vs_validation_loss.png',dpi=220,bbox_inches='tight');plt.show()
# Pair historical train snapshot with validation points nearest in optimizer update; descriptive only.
pairs=pd.merge_asof(valid_hist[['num_updates','loss']].rename(columns={'loss':'valid_loss'}),train_hist[['num_updates','loss']].rename(columns={'loss':'train_loss'}),on='num_updates',direction='nearest')
pairs['valid_minus_train']=pairs.valid_loss-pairs.train_loss
pairs.to_csv(history_dir/'H200_train_validation_gap.csv',index=False)
fig,ax=plt.subplots(figsize=(10,3.2));ax.axhline(0,color='gray',linewidth=.8)
ax.plot(pairs.num_updates,pairs.valid_minus_train,marker='.',linewidth=1)
ax.set(xlabel='Optimizer updates',ylabel='Validation − training loss',title='Descriptive train–validation gap (non-identical aggregation)');ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(history_dir/'H200_train_validation_loss_gap.png',dpi=220,bbox_inches='tight');plt.show()
display(pairs.tail(8).round(4))
print('Lowest recorded validation loss:',valid_hist.loss.min(),'at update',int(valid_hist.loc[valid_hist.loss.idxmin(),'num_updates']))
print('CAUTION: no-overfitting is not established by this plot alone. Check train/valid leakage and matched evaluation settings.')

## 2. Check ligand source counts and run quick preview

Read-only numeric LMDB keys. **Preview duplication does not estimate entire 18.84M unique count.**

In [ ]:
import lmdb, pickle
from collections import Counter

PREVIEW_TRAIN_N = 20000  # quick overview; still not a global estimate
PREVIEW_VALID_N = 3000
PREVIEW_SEED = 17

def sample_numeric_lmdb(lmdb_path, n, seed):
    env=lmdb.open(str(lmdb_path),subdir=False,readonly=True,lock=False,readahead=False,meminit=False,max_readers=128)
    try:
        with env.begin(write=False) as txn:
            count=txn.stat()['entries']
            ids=np.random.default_rng(seed).choice(count,size=min(n,count),replace=False)
            smis=[]; missing=0; n_confs=[]
            for i in ids:
                b=txn.get(str(int(i)).encode('ascii'))
                if b is None: raise RuntimeError(f'Missing numeric LMDB key {int(i)}')
                d=pickle.loads(b)  # Trusted local LMDB only.
                if not isinstance(d,dict): raise TypeError(f'Expected dict at {i}, got {type(d)}')
                s=d.get('smi',d.get(b'smi',''))
                if isinstance(s,bytes):s=s.decode('utf8',errors='replace')
                if not isinstance(s,str) or not s.strip(): missing+=1;s=''
                smis.append(s.strip())
                coords=d.get('coordinates',[])
                n_confs.append(len(coords) if hasattr(coords,'__len__') else None)
        return count,ids,smis,n_confs,missing
    finally: env.close()

def sampled_duplicates(smis, policy):
    if policy=='canonical':
        from rdkit import Chem, RDLogger
        RDLogger.DisableLog('rdApp.error')
        identities=[];invalid=0
        for s in smis:
            if not s:continue
            mol=Chem.MolFromSmiles(s)
            if mol is None: invalid+=1;continue
            identities.append(Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True))
    else:
        identities=[s for s in smis if s];invalid=0
    c=Counter(identities)
    n=len(identities)
    return {'identity':policy,'usable_sampled':n,'unique_sampled':len(c),
            'duplicate_excess_sampled':n-len(c),'duplicate_percent_within_sample':100*(n-len(c))/n if n else None,
            'rdkit_invalid_sampled':invalid}

n_train,ids_train,smis_train,slots_train,miss_train=sample_numeric_lmdb(DATA_DIR/'train.lmdb',PREVIEW_TRAIN_N,PREVIEW_SEED)
n_valid,ids_valid,smis_valid,slots_valid,miss_valid=sample_numeric_lmdb(DATA_DIR/'valid.lmdb',PREVIEW_VALID_N,PREVIEW_SEED+1)
print(f'Actual LMDB counts: TRAIN {n_train:,} / VALID {n_valid:,}')
print('Stored conformer slots (sample):',Counter(slots_train).most_common(5))
report=[]
for name,ss,missing in [('train',smis_train,miss_train),('valid',smis_valid,miss_valid)]:
    for policy in ('raw','canonical'):
        report.append({'split':name,'sample_records':len(ss),'missing_smi':missing,**sampled_duplicates(ss,policy)})
display(pd.DataFrame(report))
print('NOTE: these duplicate percentages refer ONLY to collisions within the selected sample.')

## 3. Embedded full ligand audit implementation (original logic)

The next cell contains the former `audit_lmdb.py` source directly; it uses read-only LMDB, stable SMILES digests, external `sort/comm` **within `/project`**, and counts within-train duplicates and train/valid overlap. No standalone script is created.

In [ ]:
"""Read-only molecular deduplication & train/valid overlap audit for HEART LMDB.

Examples:
  python audit_lmdb.py inspect --train /scratch/.../train.lmdb --valid /scratch/.../valid.lmdb
  python audit_lmdb.py run --train /scratch/.../train.lmdb --valid /scratch/.../valid.lmdb --out /scratch/.../audit --identity raw
  python audit_lmdb.py run --train ... --valid ... --out .../smoke --identity raw --limit 5000

The LMDBs are never written. All intermediate/output files go under --out.
A cryptographic digest is used as a compact molecular key; collisions are extraordinarily unlikely.
"""
import argparse
import hashlib
import json
import os
import pickle
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
SMILES_KEYS = ('smi', 'smiles', 'SMILES', 'canonical_smiles', 'canonical_smi')

def audit_stamp() -> str:
    return datetime.now(timezone.utc).isoformat()

def audit_open_env(path: Path):
    import lmdb
    if not path.exists():
        raise FileNotFoundError(f'LMDB missing: {path}')
    return lmdb.open(str(path), subdir=path.is_dir(), readonly=True, lock=False, readahead=False, meminit=False, max_readers=256)

def audit_extract_smi(obj) -> str | None:
    if not isinstance(obj, dict):
        return None
    for key in SMILES_KEYS:
        item = obj.get(key, obj.get(key.encode()))
        if isinstance(item, bytes):
            item = item.decode('utf-8', errors='replace')
        if isinstance(item, str) and item.strip():
            return item.strip()
    return None

def audit_inspect_one(label: str, path: Path, n: int=5) -> None:
    env = audit_open_env(path)
    try:
        stat = env.stat()
        print(f'\n== {label}: {path} ==')
        print(f"LMDB stat entries: {stat.get('entries'):,} | file/directory: {('directory' if path.is_dir() else 'file')}")
        with env.begin(write=False) as txn:
            for i, (key, value) in enumerate(txn.cursor()):
                if i >= n:
                    break
                try:
                    obj = pickle.loads(value)
                    smi = audit_extract_smi(obj)
                    ks = list(obj.keys())[:15] if isinstance(obj, dict) else None
                    conformers = obj.get('coordinates') if isinstance(obj, dict) else None
                    nconf = len(conformers) if isinstance(conformers, (tuple, list)) else None
                    print(f'  row={i} key={key[:50]!r} type={type(obj).__name__} fields={ks} n_conformers={nconf} smiles={str(smi)[:150]}')
                except Exception as e:
                    print(f'  row={i} key={key[:50]!r} decode_error={type(e).__name__}: {e}')
    finally:
        env.close()

def audit_identity_for(smi: str, method: str, chem) -> str | None:
    if method == 'raw':
        return smi
    mol = chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

def audit_export_one(label: str, lmdb_path: Path, out: Path, method: str, limit: int, report_every: int) -> dict:
    out.mkdir(parents=True, exist_ok=True)
    dest = out / f'{label}_{method}_hashes.txt'
    stats_path = out / f'{label}_{method}_export.json'
    if dest.exists() and stats_path.exists():
        existing = json.loads(stats_path.read_text())
        if existing.get('identity') != method or existing.get('limited') != bool(limit) or int(existing.get('limit', -1)) != limit or (Path(existing.get('path', '')) != lmdb_path.resolve()):
            raise ValueError(f'Existing {label} export provenance does not match current input: {stats_path}')
        expected_bytes = int(existing['usable_records']) * 33
        if dest.stat().st_size != expected_bytes:
            raise ValueError(f'Existing {label} export size is incomplete: {dest}')
        print(f'REUSE verified complete {label} export: {dest}', flush=True)
        return existing
    if dest.exists() or stats_path.exists():
        raise FileExistsError(f'Incomplete output for {label}: {dest}, {stats_path}; inspect before resuming.')
    temp = out / f'{label}_{method}_hashes.txt.part'
    if temp.exists():
        raise FileExistsError(f'Interrupted intermediate file: {temp}. Delete only that .part file after inspecting it, then rerun.')
    chem = None
    if method == 'canonical':
        from rdkit import Chem
        from rdkit import RDLogger
        RDLogger.DisableLog('rdApp.error')
        chem = Chem
    env = audit_open_env(lmdb_path)
    rows = usable = missing = invalid = decode_errors = 0
    try:
        with temp.open('w', buffering=1024 * 1024, encoding='ascii') as fout, env.begin(write=False) as txn:
            for _, value in txn.cursor():
                if limit and rows >= limit:
                    break
                rows += 1
                try:
                    obj = pickle.loads(value)
                except Exception:
                    decode_errors += 1
                    continue
                smi = audit_extract_smi(obj)
                if not smi:
                    missing += 1
                    if rows <= 20 and missing == rows:
                        print(f'WARNING: first {rows} {label} rows have no SMILES; inspect schema before full run', file=sys.stderr)
                    continue
                ident = audit_identity_for(smi, method, chem)
                if ident is None:
                    invalid += 1
                    continue
                digest = hashlib.blake2b(ident.encode('utf8'), digest_size=16).hexdigest()
                fout.write(digest + '\n')
                usable += 1
                if report_every and rows % report_every == 0:
                    print(f'{audit_stamp()} {label} rows={rows:,} usable={usable:,} missing={missing:,} invalid={invalid:,}', flush=True)
        if rows and usable == 0:
            raise RuntimeError(f'No usable SMILES in {label}; audit schema with inspect mode')
        temp.replace(dest)
        stats = {'split': label, 'path': str(lmdb_path.resolve()), 'identity': method, 'total_records_scanned': rows, 'usable_records': usable, 'missing_smiles_records': missing, 'invalid_smiles_records': invalid, 'pickle_decode_errors': decode_errors, 'limited': bool(limit), 'limit': limit, 'written_utc': audit_stamp()}
        stats_path.write_text(json.dumps(stats, indent=2) + '\n')
        print(f'DONE export {label}: {rows:,} records; {usable:,} molecules with valid identifiers. Written {dest}', flush=True)
        return stats
    finally:
        env.close()

def audit_nlines(path: Path) -> int:
    with path.open('rb') as f:
        return sum((block.count(b'\n') for block in iter(lambda: f.read(8 * 1024 * 1024), b'')))

def audit_external_sort(source: Path, sorted_path: Path, out: Path, threads: int, buffer: str) -> None:
    env = dict(os.environ)
    env['LC_ALL'] = 'C'
    temp = Path(str(sorted_path) + '.part')
    if temp.exists():
        raise FileExistsError(f'Interrupted sort output exists: {temp}. Inspect/remove only this .part file before retrying.')
    cmd = ['sort', '-u', f'--parallel={threads}', '-S', buffer, '-T', str(out), '-o', str(temp), str(source)]
    print('Sorting:', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env)
    temp.replace(sorted_path)

def audit_summarize(out: Path, method: str, threads: int, buffer: str) -> None:
    out = out.resolve()
    x = {}
    for name in ('train', 'valid'):
        raw = out / f'{name}_{method}_hashes.txt'
        stat_path = out / f'{name}_{method}_export.json'
        if not raw.is_file() or not stat_path.is_file():
            raise FileNotFoundError(f'Missing export: {raw} or {stat_path}')
        stat = json.loads(stat_path.read_text())
        if stat['identity'] != method:
            raise ValueError('Identity modes do not match')
        if audit_nlines(raw) != stat['usable_records']:
            raise ValueError(f'{raw} line count differs from export summary')
        sorted_file = out / f'{name}_{method}_unique_sorted.txt'
        if not sorted_file.exists():
            audit_external_sort(raw, sorted_file, out, threads, buffer)
        x[name] = {'stats': stat, 'unique': audit_nlines(sorted_file), 'sorted': sorted_file}
    overlap_file = out / f'train_valid_{method}_overlap_unique_hashes.txt'
    cmd = ['comm', '-12', str(x['train']['sorted']), str(x['valid']['sorted'])]
    overlap_temp = Path(str(overlap_file) + '.part')
    if overlap_temp.exists():
        raise FileExistsError(f'Interrupted overlap output exists: {overlap_temp}; inspect before resuming.')
    with overlap_temp.open('w', encoding='ascii') as f:
        subprocess.run(cmd, stdout=f, check=True, env={**os.environ, 'LC_ALL': 'C'})
    overlap_temp.replace(overlap_file)
    overlap_unique = audit_nlines(overlap_file)
    with overlap_file.open(encoding='ascii') as f:
        overlap_set = {s.strip() for s in f if s.strip()}
    with (out / f'valid_{method}_hashes.txt').open(encoding='ascii') as f:
        overlap_valid_rows = sum((row.strip() in overlap_set for row in f))
    ntr = x['train']['stats']['usable_records']
    nva = x['valid']['stats']['usable_records']
    u_tr = x['train']['unique']
    u_va = x['valid']['unique']
    result = {'method': method, 'generated_utc': audit_stamp(), 'hash_algorithm': 'BLAKE2b-128 of raw or isomeric canonical SMILES', 'train': {**x['train']['stats'], 'unique_identifiers': u_tr, 'duplicate_record_excess': ntr - u_tr, 'duplicate_record_excess_fraction': (ntr - u_tr) / ntr if ntr else None}, 'valid': {**x['valid']['stats'], 'unique_identifiers': u_va, 'duplicate_record_excess': nva - u_va, 'duplicate_record_excess_fraction': (nva - u_va) / nva if nva else None}, 'train_valid_overlap': {'overlapping_unique_identifiers': overlap_unique, 'fraction_valid_unique_identifiers_seen_in_train': overlap_unique / u_va if u_va else None, 'overlapping_validation_records': overlap_valid_rows, 'fraction_validation_records_seen_in_train': overlap_valid_rows / nva if nva else None}, 'limitations': ['If method=raw, equivalent molecules with different SMILES strings are not counted as duplicates.', 'Canonical isomeric SMILES do not merge distinct stereoisomers, tautomers, salts, or protonation states.', 'Within-split duplicate_record_excess is total usable records minus unique identifiers, not the count of unique duplicated identities.', 'If LMDB includes multiple conformers inside one record, those conformers are not separate molecular records.', 'Exact overlap alone does not detect scaffold-level similarity or establish model generalization.', 'Partial audits (--limit) are NOT representative full-dataset overlap estimates.']}
    dest = out / f'audit_summary_{method}.json'
    dest.write_text(json.dumps(result, indent=2) + '\n')
    print(f'\n=== HEART LMDB AUDIT ({method.upper()}) ===')
    for sp, dat in [('TRAIN', result['train']), ('VALID', result['valid'])]:
        print(f"{sp}: records={dat['total_records_scanned']:,} usable={dat['usable_records']:,} unique={dat['unique_identifiers']:,} duplicate_excess={dat['duplicate_record_excess']:,}")
    o = result['train_valid_overlap']
    print(f"train/valid overlap: {o['overlapping_unique_identifiers']:,} unique identifiers; {o['fraction_valid_unique_identifiers_seen_in_train'] * 100:.4f}% of valid unique IDs")
    print(f"train/valid overlap: {o['overlapping_validation_records']:,} validation records; {o['fraction_validation_records_seen_in_train'] * 100:.4f}% of valid records")
    if result['train']['limited'] or result['valid']['limited']:
        print('CAUTION: this was a LIMITED smoke audit. Values do not represent full data.')
    print(f'Saved: {dest}')

def audit_main(args_in):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='action', required=True)
    pi = sub.add_parser('inspect', help='Read and display LMDB schema and first few records only')
    pi.add_argument('--train', type=Path, required=True)
    pi.add_argument('--valid', type=Path, required=True)
    pi.add_argument('--n', type=int, default=5)
    pr = sub.add_parser('run', help='Export hash identifiers and compare duplicate/overlap counts')
    pr.add_argument('--train', type=Path, required=True)
    pr.add_argument('--valid', type=Path, required=True)
    pr.add_argument('--out', type=Path, required=True)
    pr.add_argument('--identity', choices=['raw', 'canonical'], default='raw')
    pr.add_argument('--limit', type=int, default=0, help='SMOKE only: first N records in EACH split; 0=all')
    pr.add_argument('--report-every', type=int, default=500000)
    pr.add_argument('--threads', type=int, default=4)
    pr.add_argument('--sort-buffer', default='2G')
    ps = sub.add_parser('summarize', help='Recompute summary from existing exported hash files')
    ps.add_argument('--out', type=Path, required=True)
    ps.add_argument('--identity', choices=['raw', 'canonical'], default='raw')
    ps.add_argument('--threads', type=int, default=4)
    ps.add_argument('--sort-buffer', default='2G')
    args = args_in
    if args.action == 'inspect':
        audit_inspect_one('train', args.train, args.n)
        audit_inspect_one('valid', args.valid, args.n)
    elif args.action == 'run':
        if args.limit < 0 or args.threads < 1:
            p.error('--limit must be >=0 and --threads >=1')
        args.out.mkdir(parents=True, exist_ok=True)
        for path in (args.train, args.valid):
            if path.is_dir() and args.out.resolve().is_relative_to(path.resolve()):
                p.error('Output directory must not be inside a source LMDB directory')
        audit_export_one('train', args.train, args.out, args.identity, args.limit, args.report_every)
        audit_export_one('valid', args.valid, args.out, args.identity, args.limit, args.report_every)
        audit_summarize(args.out, args.identity, args.threads, args.sort_buffer)
    else:
        audit_summarize(args.out, args.identity, args.threads, args.sort_buffer)

### 3a. Full duplicate / train–validation leakage audit

Set `RUN_FULL_RAW_AUDIT=True` when ready to scan **all** train and valid records. Canonical RDKit normalization is optional and slower. Existing completed summaries are reused.

In [ ]:
RUN_FULL_RAW_AUDIT = False       # Set True explicitly to compute the full 18.84M raw duplicate rate.
RUN_FULL_CANONICAL_AUDIT = False # Optional much more expensive RDKit canonical audit.
AUDIT_THREADS = 4
AUDIT_SORT_RAM = '512M'
LUMI_GENERAL_DISTINCT = 13_369_320

AUDIT_DIR.mkdir(parents=True,exist_ok=True)
for identity, do_run in [('raw',RUN_FULL_RAW_AUDIT),('canonical',RUN_FULL_CANONICAL_AUDIT)]:
    out=AUDIT_DIR/identity
    result=out/f'audit_summary_{identity}.json'
    if result.is_file():
        print('EXISTING completed audit:',result,'(not re-running)')
    elif do_run:
        print('Auditing directly in the current notebook — NO sbatch:',identity,flush=True)
        audit_main(Namespace(action='run',train=DATA_DIR/'train.lmdb',valid=DATA_DIR/'valid.lmdb',out=out,identity=identity,limit=0,threads=AUDIT_THREADS,sort_buffer=AUDIT_SORT_RAM,report_every=500000))
    else:
        print(f'{identity}: full audit not yet performed; enable RUN_FULL_{identity.upper()}_AUDIT to compute it.')

comparison=[]
for identity in ('raw','canonical'):
    result=AUDIT_DIR/identity/f'audit_summary_{identity}.json'
    if not result.is_file(): continue
    obj=json.loads(result.read_text())
    t,v,ov=obj['train'],obj['valid'],obj['train_valid_overlap']
    # Never label a partial result as exact 18.84M.
    if t['limited'] or v['limited'] or t['total_records_scanned']!=n_train or v['total_records_scanned']!=n_valid:
        print('PARTIAL/UNMATCHED scan, not showing as full:',result)
        continue
    comparison.append({
        'identity':identity,'train_records':t['total_records_scanned'],
        'usable_train_records':t['usable_records'],'unique_train_molecules':t['unique_identifiers'],
        'duplicate_record_excess':t['duplicate_record_excess'],
        'duplicate_rate_pct_of_usable':100*t['duplicate_record_excess_fraction'],
        'valid_records_seen_in_train':ov['overlapping_validation_records'],
        'valid_overlap_pct_of_usable':100*ov['fraction_validation_records_seen_in_train'],
        'difference_vs_LUMI_13_369_320':t['unique_identifiers']-LUMI_GENERAL_DISTINCT,
    })
if comparison:
    cmp=pd.DataFrame(comparison);display(cmp)
    fig,ax=plt.subplots(figsize=(7,4))
    vals=[LUMI_GENERAL_DISTINCT]+[z['unique_train_molecules'] for z in comparison]
    labels=['LUMI reported']+[f'HEART {z["identity"]}' for z in comparison]
    ax.bar(labels, np.array(vals)/1e6)
    ax.set(ylabel='Distinct molecules (millions)',title='Dataset-size comparison (not proof of identical dataset)')
    plt.xticks(rotation=12);plt.tight_layout();plt.show()
    print('A numerical match to ~13.37M is NOT a reproduction claim without verifying data filtering and identity definitions.')
else:
    print(f'Full unique count: UNKNOWN. LUMI reference = {LUMI_GENERAL_DISTINCT:,}.')
    print('The 18.84M records alone cannot tell us whether deduplication yields ~13.37M.')

## 4. Embedded numeric-key train + validation manifests (original validation 192 + 9,808)

This is the former `prepare_subset.py` algorithm inline. The first 192 **numeric indices** exactly match the legacy H200 `default_rng(17).choice(98844,192,replace=False)`.

In [ ]:
"""Deterministically sample HEART validation records using *numeric* LMDB keys.

CRITICAL: the HEART LMDBDataset uses txn.get(str(index).encode('ascii')).
LMDB cursor ordering is lexicographic, not numeric. Never use cursor-position
indices for SMILES metadata, because they will not match model dataset[i].

Read-only source LMDB; writes a new manifest under --out.
"""
import argparse
from datetime import datetime, timezone
import hashlib
import json
import pickle
from pathlib import Path

def subset_extract_smi(obj):
    for k in ('smi', 'smiles', 'SMILES', 'canonical_smiles'):
        if not isinstance(obj, dict):
            break
        val = obj.get(k, obj.get(k.encode()))
        if isinstance(val, bytes):
            val = val.decode('utf8', errors='replace')
        if isinstance(val, str) and val.strip():
            return val.strip()
    return ''

def subset_main(args_in):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--lmdb', type=Path, required=True)
    p.add_argument('--out', type=Path, required=True)
    p.add_argument('--n', type=int, default=10000)
    p.add_argument('--seed', type=int, default=17)
    p.add_argument('--legacy-n', type=int, default=192)
    args = args_in
    import lmdb
    import numpy as np
    import pandas as pd
    if not args.lmdb.is_file():
        p.error(f'HEART expects a single LMDB file; not found: {args.lmdb}')
    args.out.mkdir(parents=True, exist_ok=True)
    dest = args.out / 'sample_metadata.csv'
    info = args.out / 'sampling_summary.json'
    if dest.exists() or info.exists():
        p.error(f'Manifest already exists in {args.out}: use it, or choose a new output directory.')
    env = lmdb.open(str(args.lmdb), subdir=False, readonly=True, lock=False, readahead=False, meminit=False, max_readers=256)
    try:
        with env.begin() as tx:
            total = tx.stat()['entries']
            if total < 1 or not 0 < args.legacy_n <= args.n <= total:
                p.error('Require 0 < legacy-n <= n <= LMDB entry count')
            for i in [0, total - 1]:
                if tx.get(str(i).encode('ascii')) is None:
                    raise RuntimeError('LMDB is not contiguous numeric-index keyed. Do not proceed.')
        rng = np.random.default_rng(args.seed)
        first = rng.choice(total, args.legacy_n, replace=False).astype(np.int64)
        remaining = np.setdiff1d(np.arange(total, dtype=np.int64), first, assume_unique=True)
        add = rng.choice(remaining, args.n - args.legacy_n, replace=False)
        indices = np.r_[first, add]
        assert len(np.unique(indices)) == args.n
        rows = []
        with env.begin() as tx:
            for rank, idx in enumerate(indices):
                key = str(int(idx)).encode('ascii')
                raw = tx.get(key)
                if raw is None:
                    raise RuntimeError(f'Numeric key absent: {key!r}')
                rec = pickle.loads(raw)
                rows.append({'sample_order': rank, 'lmdb_index': int(idx), 'lmdb_key': key.decode(), 'legacy_192': rank < args.legacy_n, 'smiles': subset_extract_smi(rec)})
                if (rank + 1) % 2000 == 0:
                    print('Read', rank + 1, '/', len(indices), flush=True)
    finally:
        env.close()
    frame = pd.DataFrame(rows)
    frame.to_csv(dest, index=False)
    payload = {'created_utc': datetime.now(timezone.utc).isoformat(), 'source_lmdb': str(args.lmdb.resolve()), 'source_lmdb_records': total, 'n': args.n, 'random_seed': args.seed, 'legacy_n': args.legacy_n, 'selection_policy': 'legacy 192 rng.choice numeric indices + 9808 nonoverlap random numeric indices', 'index_mapping': 'numeric LMDB key str(index) — exactly as HEART LMDBDataset.__getitem__', 'selected_index_sha256': hashlib.sha256(indices.astype('<i8').tobytes()).hexdigest(), 'records_with_smiles': int(frame.smiles.astype(bool).sum())}
    info.write_text(json.dumps(payload, indent=2) + '\n')
    print(f'OK: {args.n:,} records; numeric key correctly matches task dataset[i].')
    print(dest)

In [ ]:
N_EVAL = 10000
SAMPLE_SEED = 17
MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
manifest_csv=MANIFEST_DIR/'sample_metadata.csv'
manifest_info=MANIFEST_DIR/'sampling_summary.json'
if manifest_csv.is_file() and manifest_info.is_file():
    print('Reusing existing 10,000-molecule sample:',manifest_csv)
elif manifest_csv.exists() or manifest_info.exists():
    raise RuntimeError('Only part of the manifest exists. Inspect it; this notebook will not overwrite evidence.')
else:
    subset_main(Namespace(lmdb=DATA_DIR/'valid.lmdb',out=MANIFEST_DIR,n=N_EVAL,seed=SAMPLE_SEED,legacy_n=192))
manifest=pd.read_csv(manifest_csv,keep_default_na=False)
mi=json.loads(manifest_info.read_text())
assert len(manifest)==N_EVAL and manifest['lmdb_index'].nunique()==N_EVAL
assert mi['source_lmdb_records']==n_valid
assert (manifest['lmdb_index'].astype(str)==manifest['lmdb_key'].astype(str)).all(), 'Incorrect LMDB numeric mapping'
legacy_reference=np.random.default_rng(SAMPLE_SEED).choice(n_valid,192,replace=False)
assert np.array_equal(manifest.lmdb_index.iloc[:192].to_numpy(),legacy_reference), '192-ID historical sample was not preserved'
display(manifest.head(5))
print(f'10,000 / {n_valid:,} distinct validation record indices, first 192 unchanged.')

In [ ]:
# Independent 10k training record manifest for frozen-model comparison.
TRAIN_MANIFEST_DIR=RESULTS/'training_10k_seed23'
TRAIN_MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
if (TRAIN_MANIFEST_DIR/'sample_metadata.csv').is_file() and (TRAIN_MANIFEST_DIR/'sampling_summary.json').is_file():
    print('Reusing training manifest:',TRAIN_MANIFEST_DIR)
elif any(TRAIN_MANIFEST_DIR.iterdir()):
    raise RuntimeError('Partial training manifest: inspect folder, do not overwrite.')
else:
    subset_main(Namespace(lmdb=DATA_DIR/'train.lmdb',out=TRAIN_MANIFEST_DIR,n=10000,seed=23,legacy_n=192))
train_manifest=pd.read_csv(TRAIN_MANIFEST_DIR/'sample_metadata.csv')
assert len(train_manifest)==10000 and train_manifest.lmdb_index.nunique()==10000
print('Train manifest:',len(train_manifest),'unique record indices, raw data untouched')

## 5. Embedded original HEART checkpoint evaluator

This is the existing evaluator inlined, preserving `task.build_loss`, strict checkpoint loading and the original `dataset.collater` batching. It captures raw CLS, projected CLS, masked atom/geometry/NT-Xent metrics. It **does not** initialize a random comparison or train anything.

In [ ]:
"""Read-only HEART checkpoint diagnostics and paired-view molecular embeddings.

Scientific objectives (same HEART model + original task.build_loss):
- masked atom: exact token accuracy over MASKED positions, not overall atoms;
- geometry: coord/distance original losses, plus denoising vs noisy-input baseline;
- NT-Xent: original criterion loss, positive/negative cosine, in-batch top1;
- representations: raw encoder CLS and projected normalized CLS (both views).

Uses HEART numeric-key manifest; checks source SMILES vs task sample names.
Only trusted checkpoints: torch.load(...,weights_only=False) unpickles the checkpoint.
"""
import argparse
import csv
from collections import Counter
from datetime import datetime, timezone
import hashlib
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
import numpy as np

def heart_eval_now():
    return datetime.now(timezone.utc).isoformat()

def heart_eval_sha256_file(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(4 << 20), b''):
            h.update(block)
    return h.hexdigest()

def heart_eval_move(x, device):
    import torch
    if isinstance(x, torch.Tensor):
        return x.to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: heart_eval_move(v, device) for k, v in x.items()}
    if isinstance(x, tuple):
        return tuple((heart_eval_move(v, device) for v in x))
    if isinstance(x, list):
        return [heart_eval_move(v, device) for v in x]
    return x

def heart_eval_float_scalar(x):
    import torch
    if isinstance(x, torch.Tensor):
        return float(x.detach().item()) if x.numel() == 1 else None
    if isinstance(x, (float, int, np.number)):
        return float(x)
    return None

def heart_eval_git_info(repo):
    result = {}
    try:
        for name, args in {'head': ['git', '-C', str(repo), 'rev-parse', 'HEAD'], 'status': ['git', '-C', str(repo), 'status', '--porcelain']}.items():
            result[name] = subprocess.run(args, text=True, capture_output=True, timeout=10, check=True).stdout.strip()
    except Exception as exc:
        result['error'] = str(exc)
    return result

def heart_eval_avg(x):
    return float(np.mean(x)) if len(x) else None

def heart_eval_main(args_in):
    ap = argparse.ArgumentParser(description=__doc__)
    for name in ['repo', 'data-dir', 'checkpoint', 'manifest-dir', 'out']:
        ap.add_argument('--' + name, required=True, type=Path)
    ap.add_argument('--unicore-dir', type=Path)
    ap.add_argument('--limit', type=int, default=0, help='0 = entire fixed manifest; 8 = smoke test; 192 = paired baseline comparison')
    ap.add_argument('--batch-size', type=int, default=16)
    ap.add_argument('--seed', type=int, default=13, help='Random seed for model baseline (not data sampling)')
    ap.add_argument('--variant', choices=['trained', 'random_init'], default='trained')
    ap.add_argument('--cpu', action='store_true')
    ap.add_argument('--skip-embeddings', action='store_true', help='Keep objective diagnostics only')
    args = args_in
    if args.batch_size < 2 or args.limit < 0:
        ap.error('Require batch-size>=2 for contrastive negatives and limit>=0')
    expected = args.repo / 'model' / 'unimol'
    for path in (expected, args.data_dir / f'{args.split}.lmdb', args.data_dir / 'dict.txt', args.manifest_dir / 'sample_metadata.csv', args.manifest_dir / 'sampling_summary.json'):
        if not path.exists():
            ap.error(f'Missing {path}')
    if args.variant == 'trained' and (not args.checkpoint.is_file()):
        ap.error(f'Checkpoint missing: {args.checkpoint}')
    args.out.mkdir(parents=True, exist_ok=True)
    for p in ['objective_batches.csv', 'molecule_diagnostics.csv', 'evaluation_summary.json']:
        if (args.out / p).exists():
            ap.error(f'Output {args.out / p} already exists. Use separate run folder to avoid overwritten evidence.')
    if args.unicore_dir and args.unicore_dir.is_dir():
        sys.path.insert(0, str(args.unicore_dir.resolve()))
    sys.path.insert(0, str((args.repo / 'model').resolve()))
    import pandas as pd
    import torch
    from unicore import tasks
    import unicore
    import unimol
    print('unimol:', unimol.__file__, 'unicore:', unicore.__file__, flush=True)
    if expected.resolve() not in Path(unimol.__file__).resolve().parents:
        raise RuntimeError('Wrong UniMol module imported: ' + unimol.__file__)
    device = torch.device('cpu' if args.cpu else 'cuda:0')
    if device.type == 'cuda' and (not torch.cuda.is_available()):
        raise RuntimeError('No CUDA device. Run within GPU Slurm allocation or --cpu for small smoke.')
    torch.set_num_threads(min(16, max(1, int(os.environ.get('SLURM_CPUS_PER_TASK', '8')))))
    print('device:', device, 'variant:', args.variant, flush=True)
    checkpoint = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
    if 'model' not in checkpoint or 'args' not in checkpoint:
        raise RuntimeError('Expected full original Uni-Core checkpoint with model and args')
    from argparse import Namespace
    cp_args = checkpoint['args']
    if isinstance(cp_args, dict):
        cp_args = Namespace(**cp_args)
    if not hasattr(cp_args, 'task') or not hasattr(cp_args, 'loss'):
        raise RuntimeError('Checkpoint args lack task/loss. Inspect real checkpoint schema.')
    cp_args.data = str(args.data_dir.resolve())
    cp_args.mode = 'train'
    if hasattr(cp_args, 'num_workers'):
        cp_args.num_workers = 0
    if hasattr(cp_args, 'user_dir'):
        cp_args.user_dir = str(expected.resolve())
    task = tasks.setup_task(cp_args)
    if args.variant == 'random_init':
        torch.manual_seed(args.seed)
        np.random.seed(args.seed)
    model = task.build_model(cp_args)
    if args.variant == 'trained':
        load_info = model.load_state_dict(checkpoint['model'], strict=True)
        print('Checkpoint load (strict):', load_info, flush=True)
    criterion = task.build_loss(cp_args)
    model = model.to(device).eval()
    criterion = criterion.to(device).eval()
    del checkpoint
    task.load_dataset(args.split, epoch=1)
    dataset = task.dataset(args.split)
    if hasattr(dataset, 'set_epoch'):
        dataset.set_epoch(1)
    meta = pd.read_csv(args.manifest_dir / 'sample_metadata.csv', keep_default_na=False)
    info = json.loads((args.manifest_dir / 'sampling_summary.json').read_text())
    if len(meta) != info['n'] or len(dataset) != info['source_lmdb_records']:
        raise RuntimeError('Manifest length / dataset count mismatch!')
    if not np.array_equal(meta.sample_order.to_numpy(), np.arange(len(meta))):
        raise RuntimeError('Manifest order invalid')
    if not (meta.lmdb_index.astype(str) == meta.lmdb_key.astype(str)).all():
        raise RuntimeError('Non-numeric manifest LMDB mapping; recreate using NEW prepare_subset.py')
    if args.limit:
        meta = meta.iloc[:args.limit].copy()
    if meta.empty:
        raise RuntimeError('No records')
    if meta.lmdb_index.min() < 0 or meta.lmdb_index.max() >= len(dataset):
        raise RuntimeError('Invalid LMDB index in manifest')
    N = len(meta)
    dims = int(getattr(cp_args, 'encoder_embed_dim', 512))
    maps = {}
    if not args.skip_embeddings:
        for name in ['cls_view0', 'cls_view1', 'projected_view0', 'projected_view1']:
            maps[name] = np.lib.format.open_memmap(args.out / f'{name}.npy', mode='w+', dtype='float32', shape=(N, dims))
    captures = {}

    def enc_hook(mod, inputs, output):
        enc = output[0] if isinstance(output, (tuple, list)) else output
        captures['encoder_cls'] = enc[:, 0, :].detach()

    def model_hook(mod, inputs, output):
        if not isinstance(output, (tuple, list)) or len(output) != 6:
            raise RuntimeError('Unexpected HEART model return structure')
        captures['logits'] = output[0].detach() if output[0] is not None else None
        captures['distance'] = output[1].detach() if output[1] is not None else None
        captures['coord'] = output[2].detach() if output[2] is not None else None
        captures['projected'] = output[5].detach()
    h1 = model.encoder.register_forward_hook(enc_hook)
    h2 = model.register_forward_hook(model_hook)
    batch_fields = ['start', 'n_molecules', 'n_views', 'masked_token_cnt', 'masked_token_hit', 'masked_acc', 'masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'official_weighted_loss', 'coord_pred_rmse', 'coord_noisy_rmse', 'coord_denoising_improvement', 'distance_pred_rmse_original_units', 'positive_projection_cosine', 'negative_projection_cosine', 'contrastive_top1_view0_to_view1', 'contrastive_top1_view1_to_view0', 'augmentation_token_mismatch_fraction']
    mol_fields = ['sample_order', 'lmdb_index', 'view', 'masked_atom_count', 'masked_atom_hits', 'masked_atom_accuracy', 'coord_pred_rmse', 'coord_noisy_rmse']
    collected = []
    masked_hits = masked_counts = 0
    atom_target_counts = Counter()
    atom_correct_counts = Counter()
    geometry_sq = geometry_noisy_sq = geometry_nums = 0
    pos_vals = []
    neg_vals = []
    retrieval_hits = 0
    retrieval_possible = 0
    sampled_token_view_changes = []
    status = 'failed'
    try:
        with (args.out / 'objective_batches.csv').open('w', newline='') as bf, (args.out / 'molecule_diagnostics.csv').open('w', newline='') as mf:
            bw = csv.DictWriter(bf, fieldnames=batch_fields)
            bw.writeheader()
            mw = csv.DictWriter(mf, fieldnames=mol_fields)
            mw.writeheader()
            with torch.inference_mode():
                for begin in range(0, N, args.batch_size):
                    stop = min(begin + args.batch_size, N)
                    sub = meta.iloc[begin:stop]
                    ids = sub.lmdb_index.astype(int).tolist()
                    b = len(ids)
                    sample = dataset.collater([dataset[i] for i in ids])
                    if not sample:
                        raise RuntimeError('Empty batch at ' + str(begin))
                    if 'smi_name' in sample.get('target', {}):
                        got = sample['target']['smi_name']
                        expect = sub.smiles.astype(str).tolist()
                        if len(got) != 2 * b:
                            raise RuntimeError('Expected 2-batch SMILES target names')
                        for j in range(b):
                            x = got[j]
                            if isinstance(x, bytes):
                                x = x.decode('utf8', errors='replace')
                            if str(x).strip() != expect[j].strip():
                                raise RuntimeError(f'SMILES misalignment at id={ids[j]}: manifest={expect[j]!r}, task={x!r}')
                    sample = heart_eval_move(sample, device)
                    captures.clear()
                    loss, sample_size, log = criterion(model, sample)
                    if not all((k in captures for k in ('encoder_cls', 'projected', 'logits'))):
                        raise RuntimeError('Model/encoder hooks missing; check code version')
                    masked = sample['target']['tokens_target'].ne(criterion.padding_idx)
                    if masked.shape[0] != 2 * b:
                        raise RuntimeError(f'Expected two views {2 * b}, got {masked.shape[0]}')
                    target = sample['target']['tokens_target']
                    logits = captures['logits']
                    if logits is None or logits.shape[0] != int(masked.sum()):
                        raise RuntimeError('Masked logits do not align with masked positions')
                    pred = logits.argmax(dim=-1)
                    truth = target[masked]
                    total = int(masked.sum().item())
                    hit = int((pred == truth).sum().item())
                    masked_hits += hit
                    masked_counts += total
                    target_ids = truth.detach().cpu().tolist()
                    correct_ids = truth[pred == truth].detach().cpu().tolist()
                    atom_target_counts.update((int(x) for x in target_ids))
                    atom_correct_counts.update((int(x) for x in correct_ids))
                    pred_filled = torch.full_like(target, -999)
                    pred_filled[masked] = pred
                    each_cnt = masked.sum(dim=1)
                    each_hit = ((pred_filled == target) & masked).sum(dim=1)
                    coord_pred = captures.get('coord')
                    coord_target = sample['target']['coord_target']
                    src_coord = sample['net_input']['src_coord']
                    if coord_pred is not None:
                        err = (coord_pred.float() - coord_target.float()).pow(2).sum(dim=-1)
                        noisy = (src_coord.float() - coord_target.float()).pow(2).sum(dim=-1)
                        per_coord = (err * masked).sum(dim=1) / (each_cnt.clamp_min(1) * 3)
                        per_noisy = (noisy * masked).sum(dim=1) / (each_cnt.clamp_min(1) * 3)
                        per_coord_rmse = per_coord.sqrt().cpu().numpy()
                        per_noisy_rmse = per_noisy.sqrt().cpu().numpy()
                        geometry_sq += float((err * masked).sum().item())
                        geometry_noisy_sq += float((noisy * masked).sum().item())
                        geometry_nums += 3 * total
                        coord_rmse = float((err[masked].mean() / 3).sqrt().item()) if total else float('nan')
                        noisy_rmse = float((noisy[masked].mean() / 3).sqrt().item()) if total else float('nan')
                    else:
                        coord_rmse = noisy_rmse = float('nan')
                        per_coord_rmse = per_noisy_rmse = np.full(2 * b, np.nan)
                    dist_pred = captures.get('distance')
                    d_rmse = float('nan')
                    if dist_pred is not None:
                        pairmask = masked.unsqueeze(-1) & sample['net_input']['src_tokens'].ne(criterion.padding_idx).unsqueeze(1)
                        dp = dist_pred.float() * criterion.dist_std + criterion.dist_mean
                        dt = sample['target']['distance_target'].float()
                        if pairmask.any():
                            d_rmse = float((dp[pairmask] - dt[pairmask]).pow(2).mean().sqrt().item())
                    z = captures['projected']
                    cls = captures['encoder_cls']
                    if z.shape != (2 * b, dims) or cls.shape != (2 * b, dims):
                        raise RuntimeError(f'Embedding shape mismatch: cls={cls.shape}, proj={z.shape}; expected 2*b x {dims}')
                    z0 = z[:b].float()
                    z1 = z[b:].float()
                    similarities = z0 @ z1.T
                    same = similarities.diag()
                    pos_vals.extend(same.detach().cpu().tolist())
                    negative = similarities[~torch.eye(b, dtype=torch.bool, device=device)]
                    neg = negative.mean().item() if negative.numel() else float('nan')
                    if negative.numel():
                        neg_vals.append(float(neg))
                    top01 = float((similarities.argmax(dim=1) == torch.arange(b, device=device)).float().mean().item())
                    top10 = float((similarities.argmax(dim=0) == torch.arange(b, device=device)).float().mean().item())
                    retrieval_hits += int(round(top01 * b)) + int(round(top10 * b))
                    retrieval_possible += 2 * b
                    tok = sample['net_input']['src_tokens']
                    token_mismatch = float((tok[:b] != tok[b:]).float().mean().item())
                    sampled_token_view_changes.append(token_mismatch)
                    if maps:
                        for name, arr in [('cls_view0', cls[:b]), ('cls_view1', cls[b:]), ('projected_view0', z0), ('projected_view1', z1)]:
                            maps[name][begin:stop] = arr.float().cpu().numpy()
                    for v in range(2):
                        for j in range(b):
                            k = v * b + j
                            nmask = int(each_cnt[k].item())
                            ncorrect = int(each_hit[k].item())
                            mw.writerow({'sample_order': int(sub.iloc[j].sample_order), 'lmdb_index': int(ids[j]), 'view': v, 'masked_atom_count': nmask, 'masked_atom_hits': ncorrect, 'masked_atom_accuracy': ncorrect / nmask if nmask else '', 'coord_pred_rmse': float(per_coord_rmse[k]), 'coord_noisy_rmse': float(per_noisy_rmse[k])})

                    def logged(key):
                        return heart_eval_float_scalar(log.get(key))
                    row = {'start': begin, 'n_molecules': b, 'n_views': 2 * b, 'masked_token_cnt': total, 'masked_token_hit': hit, 'masked_acc': hit / total if total else None, 'masked_token_loss': logged('masked_token_loss'), 'masked_coord_loss': logged('masked_coord_loss'), 'masked_dist_loss': logged('masked_dist_loss'), 'contrastive_loss': logged('contrastive_loss'), 'official_weighted_loss': logged('loss'), 'coord_pred_rmse': coord_rmse, 'coord_noisy_rmse': noisy_rmse, 'coord_denoising_improvement': noisy_rmse - coord_rmse, 'distance_pred_rmse_original_units': d_rmse, 'positive_projection_cosine': float(same.mean().item()), 'negative_projection_cosine': float(neg), 'contrastive_top1_view0_to_view1': top01, 'contrastive_top1_view1_to_view0': top10, 'augmentation_token_mismatch_fraction': token_mismatch}
                    bw.writerow(row)
                    bf.flush()
                    mf.flush()
                    collected.append(row)
                    if stop % max(128, args.batch_size) < args.batch_size or stop == N:
                        print(f'[{args.variant}] {stop:,}/{N:,} masked_acc={masked_hits / masked_counts:.4%} coordRMSE={coord_rmse:.4f} vs noisy={noisy_rmse:.4f}', flush=True)
                    del sample, captures['encoder_cls'], captures['projected'], captures['logits']
            for m in maps.values():
                m.flush()
        status = 'complete'
    finally:
        h1.remove()
        h2.remove()
    avgs = {key: heart_eval_avg([r[key] for r in collected if r.get(key) is not None and np.isfinite(r[key])]) for key in ['masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'official_weighted_loss', 'positive_projection_cosine', 'negative_projection_cosine', 'contrastive_top1_view0_to_view1', 'contrastive_top1_view1_to_view0', 'distance_pred_rmse_original_units']}
    atom_csv = args.out / 'atom_token_class_accuracy.csv'
    symbol_names = getattr(task.dictionary, 'symbols', None) if hasattr(task, 'dictionary') else None
    with atom_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['token_id', 'token_symbol', 'masked_targets', 'correct', 'accuracy', 'target_fraction'])
        writer.writeheader()
        for token_id, n in atom_target_counts.most_common():
            symbol = str(symbol_names[token_id]) if symbol_names is not None and token_id < len(symbol_names) else str(token_id)
            correct = atom_correct_counts[token_id]
            writer.writerow({'token_id': token_id, 'token_symbol': symbol, 'masked_targets': n, 'correct': correct, 'accuracy': correct / n if n else None, 'target_fraction': n / masked_counts if masked_counts else None})
    summary = {'status': status, 'utc': heart_eval_now(), 'variant': args.variant, 'n_molecules': N, 'n_views': 2 * N, 'batches': len(collected), 'masked_tokens': masked_counts, 'masked_token_hits': masked_hits, 'masked_atom_accuracy': masked_hits / masked_counts if masked_counts else None, 'masked_atom_majority_class_baseline_accuracy': max(atom_target_counts.values()) / masked_counts if masked_counts else None, 'masked_atom_majority_class_token_id': atom_target_counts.most_common(1)[0][0] if atom_target_counts else None, 'atom_class_csv': str(atom_csv), 'coord_pred_rmse': float(np.sqrt(geometry_sq / geometry_nums)) if geometry_nums else None, 'coord_noisy_rmse': float(np.sqrt(geometry_noisy_sq / geometry_nums)) if geometry_nums else None, 'coord_rmse_reduction_fraction': float(1 - np.sqrt(geometry_sq / geometry_noisy_sq)) if geometry_noisy_sq else None, 'contrastive_top1_both_directions': retrieval_hits / retrieval_possible if retrieval_possible else None, 'positive_projection_cosine_mean': heart_eval_avg(pos_vals), 'negative_projection_cosine_mean_of_batches': heart_eval_avg(neg_vals), 'criterion_mean_over_batches': avgs, 'batch_size_molecules': args.batch_size, 'contrastive_note': 'NT-Xent is batch-size-dependent; top1 only within a batch of this size, not global 10k retrieval. The 0.624 historic validation loss at batch 448 must not be directly compared numerically against a different-batch-size 10k loss.', 'geometry_note': 'RMSE is in the coordinate units of transformed LMDB target (centering normalized, not necessarily global alignment). SmoothL1 is the training objective.', 'source_lmdb': str((args.data_dir / f'{args.split}.lmdb').resolve()), 'manifest': str((args.manifest_dir / 'sample_metadata.csv').resolve()), 'manifest_sha256': heart_eval_sha256_file(args.manifest_dir / 'sample_metadata.csv'), 'checkpoint': str(args.checkpoint.resolve()), 'checkpoint_sha256': heart_eval_sha256_file(args.checkpoint), 'checkpoint_args': {name: getattr(cp_args, name, None) for name in ['task', 'loss', 'arch', 'encoder_embed_dim', 'encoder_layers', 'encoder_attention_heads', 'contrastive_view_mode', 'mask_prob', 'noise', 'noise_type', 'masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'only_polar', 'max_atoms']}, 'model_git': heart_eval_git_info(args.repo), 'embedding_files': list(maps.keys()), 'raw_metric_definitions': {'masked_acc': 'global sum of correct predictions divided by all masked token targets (both augmented views)', 'coord_rmse': 'RMSE between predicted coords and target 3D coords at masked atoms; compare with noisy input RMSE', 'contrastive_top1': 'in-batch diagonal pairing rank-1 for projected normalized representations', 'original_loss': 'batch-mean of original loss outputs; contrastive scale depends on batch size'}, 'warning': 'Pretraining-objective evaluation is not evidence of downstream lipid RNA-delivery accuracy. Data leakage must be audited separately.'}
    (args.out / 'evaluation_summary.json').write_text(json.dumps(summary, indent=2, default=str) + '\n')
    print('COMPLETE:', args.out / 'evaluation_summary.json', flush=True)

### 5a. Run 10,000 checkpoint inference (disabled until explicitly enabled)

The GPU must be available in the **same notebook kernel**. Batch 8 molecules × 2 augmented views, same as the earlier H200 embedding notebook; no Slurm.

In [ ]:
RUN_EVAL_SMOKE = False    # Optional 8 validation molecules
RUN_EVAL_10K = False      # Frozen checkpoint on both 10k training AND 10k validation
ALLOW_CPU_EVALUATION = False
BATCH_MOLECULES = 8
import torch
print('Current kernel:',sys.executable,'| torch:',torch.__version__,'| CUDA:',torch.cuda.is_available())
RUN_ROOT.mkdir(parents=True,exist_ok=True)
for label,enabled,limit,split,manifest_dir in [
    ('valid_smoke_8',RUN_EVAL_SMOKE,8,'valid',MANIFEST_DIR),
    ('train_10k',RUN_EVAL_10K,0,'train',TRAIN_MANIFEST_DIR),
    ('full_10k',RUN_EVAL_10K,0,'valid',MANIFEST_DIR),
]:
    dest=RUN_ROOT/f'{label}_batch{BATCH_MOLECULES}'
    if (dest/'evaluation_summary.json').is_file():
        print('Reusing completed result:',dest)
    elif enabled:
        if not torch.cuda.is_available() and not ALLOW_CPU_EVALUATION:
            raise RuntimeError('GPU not visible in this notebook kernel; select heart_gpu kernel / GPU Jupyter session. No job submitted.')
        print('Frozen model evaluation:',split,limit or 10000)
        heart_eval_main(Namespace(repo=REPO_V2,data_dir=DATA_DIR,checkpoint=CHECKPOINT,manifest_dir=manifest_dir,
                                  out=dest,unicore_dir=UNICORE_ROOT,limit=limit,batch_size=BATCH_MOLECULES,seed=13,
                                  variant='trained',cpu=not torch.cuda.is_available(),skip_embeddings=False,split=split))
    else:print(label,'not run. Enable flag when ready.')

## 6. Frozen checkpoint on **10k train vs 10k validation** — report comparison

The same checkpoint is evaluated on both splits, **batch size 8 / 2 views**, with the original criterion. Historical batch-448 training losses and batch-8 sampled losses cannot be directly compared. Train/valid dataset augmentations may differ; treat gaps as descriptive.

In [ ]:
train_result=RUN_ROOT/f'train_10k_batch{BATCH_MOLECULES}'/'evaluation_summary.json'
valid_result=RUN_ROOT/f'full_10k_batch{BATCH_MOLECULES}'/'evaluation_summary.json'
if train_result.is_file() and valid_result.is_file():
    trm=json.loads(train_result.read_text());vam=json.loads(valid_result.read_text())
    assert trm.get('status')=='complete' and vam.get('status')=='complete'
    assert trm['n_molecules']==vam['n_molecules']==10000
    checks=['masked_atom_accuracy','coord_pred_rmse','coord_noisy_rmse','contrastive_top1_both_directions','positive_projection_cosine_mean']
    summary_comp=pd.DataFrame({'metric':checks,'training':[trm.get(k) for k in checks],'validation':[vam.get(k) for k in checks]})
    btr=pd.read_csv(train_result.parent/'objective_batches.csv');bva=pd.read_csv(valid_result.parent/'objective_batches.csv')
    objective_keys=[x for x in ['masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','official_weighted_loss'] if x in btr and x in bva]
    loss_table=pd.DataFrame({'objective':objective_keys,'training_mean':[btr[k].mean() for k in objective_keys],'validation_mean':[bva[k].mean() for k in objective_keys]})
    loss_table['valid_minus_train']=loss_table.validation_mean-loss_table.training_mean
    display(summary_comp.round(4));display(loss_table.round(4))
    report_dir=RESULTS/'training_vs_validation_10k';report_dir.mkdir(parents=True,exist_ok=True)
    summary_comp.to_csv(report_dir/'checkpoint_10k_train_vs_valid_metrics.csv',index=False)
    loss_table.to_csv(report_dir/'checkpoint_10k_train_vs_valid_objectives.csv',index=False)
    xx=np.arange(len(loss_table));fig,ax=plt.subplots(figsize=(10,4.6))
    ax.bar(xx-.2,loss_table.training_mean,.4,label='Training 10k')
    ax.bar(xx+.2,loss_table.validation_mean,.4,label='Validation 10k')
    ax.set_xticks(xx,loss_table.objective,rotation=18,ha='right');ax.set(ylabel='Original model criterion component (batch 8)',title='Frozen H200 checkpoint: 10k train vs 10k validation')
    ax.legend();fig.tight_layout();fig.savefig(report_dir/'checkpoint_10k_train_vs_valid_losses.png',dpi=220,bbox_inches='tight');plt.show()
    print('These are checkpoint evaluation losses on sampled records, not optimizer-step training losses.')
else:print('Matched 10k training/validation comparison not yet available. Enable RUN_EVAL_10K=True in Section 5.')

In [ ]:
RUN_10K=RUN_ROOT/f'full_10k_batch{BATCH_MOLECULES}'
summary_path=RUN_10K/'evaluation_summary.json'
if not summary_path.is_file():
    display(Markdown('**10k evaluation not run yet.** Enable `RUN_EVAL_10K=True` in Section 5 when resources are available.'))
else:
    metrics=json.loads(summary_path.read_text())
    assert metrics.get('status')=='complete' and metrics.get('n_molecules')==10000,'Evaluation is incomplete or wrong-sized'
    display(pd.DataFrame([{
        'Validation molecules':metrics.get('n_molecules'),
        'Masked accuracy':metrics.get('masked_atom_accuracy'),
        'Masked majority-class baseline':metrics.get('masked_atom_majority_class_baseline_accuracy'),
        'Coordinate RMSE (pred)':metrics.get('coord_pred_rmse'),
        'Coordinate RMSE (noisy)':metrics.get('coord_noisy_rmse'),
        'RMSE improvement fraction':metrics.get('coord_rmse_reduction_fraction'),
        'NT-Xent batch top-1 retrieval':metrics.get('contrastive_top1_both_directions'),
        'Mean positive cosine':metrics.get('positive_projection_cosine_mean'),
        'Mean negative cosine':metrics.get('negative_projection_cosine_mean_of_batches')
    }]))
    batch=pd.read_csv(RUN_10K/'objective_batches.csv')
    cols=[c for c in ('masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','official_weighted_loss') if c in batch.columns]
    display(batch[cols].describe().round(4))
    mol=pd.read_csv(RUN_10K/'molecule_diagnostics.csv')
    fig, axes=plt.subplots(1,3,figsize=(13,3.5))
    mol.masked_atom_accuracy.dropna().hist(bins=25,ax=axes[0]); axes[0].set(title='Masked atom accuracy per view',xlabel='Accuracy')
    z=batch[['coord_pred_rmse','coord_noisy_rmse']].dropna()
    if len(z):
        axes[1].scatter(z.coord_noisy_rmse,z.coord_pred_rmse,alpha=.3,s=10)
        lim=max(z.max());axes[1].plot([0,lim],[0,lim],ls='--',c='gray')
    axes[1].set(title='Coordinate recovery vs noisy input',xlabel='Noisy-input RMSE',ylabel='Predicted RMSE')
    if 'contrastive_loss' in batch:batch.contrastive_loss.dropna().hist(bins=25,ax=axes[2])
    axes[2].set(title='NT-Xent loss across batches',xlabel='Loss')
    plt.tight_layout();plt.show()
    display(pd.read_csv(RUN_10K/'atom_token_class_accuracy.csv').head(15))

## 7. PCA/UMAP of 10,000 original H200 representations

In [ ]:
RUN_REPRESENTATION_PLOTS = True  # plots only when complete 10k embedding output exists
if not (RUN_10K/'cls_view0.npy').is_file():
    print('No 10k CLS array yet; run Section 5 first (RUN_EVAL_10K=True).')
elif RUN_REPRESENTATION_PLOTS:
    from sklearn.decomposition import PCA
    X=np.load(RUN_10K/'cls_view0.npy',mmap_mode='r')
    XP=np.load(RUN_10K/'projected_view0.npy',mmap_mode='r')
    assert X.shape[0]==XP.shape[0]==10000 and X.ndim==XP.ndim==2
    assert np.isfinite(X).all() and np.isfinite(XP).all()
    sample_meta=manifest[['sample_order','lmdb_index','smiles']].copy()
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, rdMolDescriptors
    RDLogger.DisableLog('rdApp.error')
    mw=[];heavy=[];clogp=[];ring=[]
    for smi in sample_meta.smiles.astype(str):
        mol=Chem.MolFromSmiles(smi)
        if mol is None: mw.append(np.nan);heavy.append(np.nan);clogp.append(np.nan);ring.append(np.nan)
        else:
            mw.append(Descriptors.MolWt(mol)); heavy.append(mol.GetNumHeavyAtoms())
            clogp.append(Descriptors.MolLogP(mol));ring.append(rdMolDescriptors.CalcNumRings(mol))
    sample_meta['MW']=mw;sample_meta['heavy_atoms']=heavy;sample_meta['cLogP']=clogp;sample_meta['rings']=ring
    print('Source molecular property coverage:',sample_meta['MW'].notna().mean())
    figure_dir=RUN_10K/'figures';figure_dir.mkdir(parents=True,exist_ok=True)
    for label,emb in [('CLS_raw',X),('Projector_NTXent',XP)]:
        # Randomized PCA avoids a very large 10k x 10k similarity matrix.
        pca=PCA(n_components=2,svd_solver='randomized',random_state=17)
        XY=pca.fit_transform(np.asarray(emb,dtype=np.float32))
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for ax,property_name in zip(axes,['heavy_atoms','cLogP']):
            col=sample_meta[property_name].to_numpy()
            sc=ax.scatter(XY[:,0],XY[:,1],c=col,s=3,alpha=.65,cmap='viridis',rasterized=True)
            ax.set(title=f'{label} PCA — {property_name}',xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                   ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
            fig.colorbar(sc,ax=ax,label=property_name)
        plt.tight_layout();plt.savefig(figure_dir/f'{label}_PCA.png',dpi=160);plt.show()
        try:
            import umap
        except ImportError:
            print('UMAP skipped: install umap-learn in existing environment if permitted. No auto-install.');continue
        reducer=umap.UMAP(n_components=2,n_neighbors=15,min_dist=.1,metric='cosine',random_state=17)
        XYu=reducer.fit_transform(np.asarray(emb,dtype=np.float32))
        assert XYu.shape==(10000,2)
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for ax,prop in zip(axes,['heavy_atoms','cLogP']):
            sc=ax.scatter(XYu[:,0],XYu[:,1],c=sample_meta[prop],s=3,alpha=.65,cmap='viridis',rasterized=True)
            ax.set(title=f'{label} UMAP — {prop}',xlabel='UMAP 1',ylabel='UMAP 2')
            fig.colorbar(sc,ax=ax,label=prop)
        plt.tight_layout();plt.savefig(figure_dir/f'{label}_UMAP.png',dpi=160);plt.show()
        pd.DataFrame({'sample_order':sample_meta.sample_order,'lmdb_index':sample_meta.lmdb_index,
                      'UMAP1':XYu[:,0],'UMAP2':XYu[:,1]}).to_csv(figure_dir/f'{label}_UMAP.csv',index=False)

## 8. Reporting interpretation

**Training curves:** show train and valid together, plus the descriptive gap. **Matched 10k train vs validation:** both sampled splits use the same frozen checkpoint, original loss and batch size 8. If validation loss does not turn upward while training improves, write **“No clear validation-loss divergence was observed in the evaluated training interval.”** Avoid **“proves no overfitting”**.

**Duplication & leakage:** a full raw-SMILES audit is required before claiming independent train/validation molecules; RDKit canonical SMILES may reveal extra duplicates, and neither proves scaffold independence. LUMI 13,369,320 distinct molecules are **a literature reference, not a target count**.

**10k evaluation:** evaluate the frozen H200 checkpoint on 10,000 training and 10,000 held-out validation records separately. Report masked-token accuracy, geometry denoising vs noisy coordinates, contrastive in-batch top1, the original objective loss components, and PCA/UMAP. Label NT-Xent **batch-size-dependent** and don't equate its batch-8 value to historic batch-448 validation loss. This is **pretraining diagnostics**, not proof of RNA delivery or lipid performance.

**Outputs:** training-history plot CSVs and PNGs in `/project/yli4/ypan/heart_evaluation/general_audit_10k_v2/training_history/`; dataset audit, 10k manifest and model plots in adjacent project result folders. The original `/home` source and `/project` inputs remain unchanged.